<h1 style="text-align: center;">The Cycloid: Exploring the Brachistochrone and Tautochrone Properties</h1>

## Content

1. [Introduction](#introduction)
2. [Importing Dependencies](#importing-dependencies)
3. [Demonstrating Brachistochrone](#brachistochrone-the-path-of-shortest-time)
    - [Drawing the paths](#connecting-two-points-with-different-paths)
    - [Calculating the Time of Descent in each path](#calculating-the-time-of-descent-in-each-path)
    - [Adjustment of ball's position on the path](#visualizing-tangent-and-normal-vectors-realistic-ball-rolling-on-a-cycloid)
    - [Animating the Brachistochrone Motion](#animating-the-brachistochrone-motion)
    - [Comparing the Brachistochrone with lines of variable slopes](#comparison-with-lines-of-different-slopes)
4. [Isochronous Property of Cycloid](#isochronous-property-of-cycloid)
5. [References](#references)

## Introduction

In this notebook, I explored two remarkable and well-known properties of the cycloid: the Brachistochrone (the path of shortest time) and its isochronous property. 

I began by deriving the equation of the Brachistochrone curve using principles of energy conservation, then simplified it using the calculus of variations and the Euler-Lagrange equations. This derivation reveals that the path of shortest time is an inverted cycloid.

Next, I connected the two endpoints of the cycloidal curve with other paths: a parabolic curve, a circular arc, and a straight line. For each of these paths, I derived its equation under the condition that it must pass through both the origin and the endpoint of the cycloid.

I demonstrated that the ball always takes the least time to reach the bottom of the cycloidal curve under the influence of gravity (without friction). Additionally, I calculated the time taken by the ball to descend along each path. For the cycloid and the straight line, I derived analytical expressions for the time of descent. However, for the parabolic and circular paths, the integration becomes more complex, so I employed numerical integration using `quad` from `scipy.integrate`.

To create the illusion of a ball rolling realistically along a curve, I offset the ball's center by a distance equal to its radius along the curve's normal. I calculated the normal vector for each path and illustrated this adjustment in an animation that compares a ball rolling down the cycloid with and without the normal offset.

After determining the descent times and accounting for the normal offset, I animated the motion of the balls rolling down various paths. The animation clearly shows that the cycloid is the path of shortest time, followed by the parabola, the circle, and finally the straight line. While the straight line represents the shortest distance, it is not the path of shortest time.

Additionally, I created an animation comparing the cycloid to lines of different slopes. Even with an increased slope, the cycloid consistently outperforms those lines in terms of descent time. A table in the figure displays the time taken for the ball to reach the endpoints of each linear path, along with the corresponding intermediate times along the cycloidal path. This table shows that the cycloid always takes less time than any line with a varying slope.

Lastly, I demonstrated the isochronous property of the cycloid, where the ball reaches the bottom at the same time regardless of its starting position on the curve. This was showcased by rolling multiple balls down the cycloidal curve from different starting positions, with all balls reaching the bottom simultaneously. I also extended this demonstration into 3D, rolling several balls on different curves, each starting from a unique initial position. It is fascinating to observe that all the balls still reach the bottom at the same time.

## Importing Dependencies

In [ ]:
# Core libraries
import os
import math
import itertools

# Numerical and scientific computing
import numpy as np
from scipy import integrate

# Visualization
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib import patches
from matplotlib import colormaps as cm
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D


# Matplotlib interactive mode
# For running animation inside jupyter notebook
%matplotlib ipympl


## Brachistochrone: The path of Shortest Time


**The Problem:**
We want to find the path $y(x)$ between two points $(0,0)$ and $(x_f, y_f)$ (where $y_f < 0$) that a particle will slide along under gravity, starting from rest at $(0,0)$, in the minimum amount of time.

**1. Coordinate System:**
Let's set up a coordinate system with the origin at the starting point $(0,0)$. We'll define the y-axis as pointing *downwards*. This means points below the start have $y < 0$. Gravity acts in the negative y direction.

**2. Energy Conservation:**
The particle starts from rest ($v_0=0$) at the origin $(0,0)$. Let's set the potential energy $U$ to be zero at $y=0$. As the particle descends to a point $(x,y)$ on the curve (where $y$ is the vertical distance descended downwards), its height is $-y$ relative to the starting level. The potential energy at $(x,y)$ is $U(y) = mg(-y) = -mgy$. However, using our convention that y points *downwards* and gravity accelerates in the -y direction, the potential energy at a depth $y$ below the origin is $U(y) = -mgy$ *if* we set $U(0)=0$. A more intuitive way with y downwards is to think of the *change* in potential energy. As the particle falls a distance $y$, the gravitational force $mg$ does positive work $mgy$. By conservation of energy, the increase in kinetic energy must equal the decrease in potential energy.

Initial Energy (at (0,0)): $E_i = \frac{1}{2}mv_0^2 + U(0) = 0 + 0 = 0$.  

Energy at point $(x,y)$: $E(x,y) = \frac{1}{2}mv^2 + U(y)$.  

Since gravity has done $mgy$ work, the potential energy at depth $y$ (relative to 0 at $y=0$) is $-mgy$. Therefore,

$E(x,y) = \frac{1}{2}mv^2 - mgy$.

By conservation of energy, $E(x,y) = E_i = 0$:
$$
\begin{aligned}
&\frac{1}{2}mv^2 - mgy = 0\\
\implies \, &\frac{1}{2}mv^2 = mgy\\
\implies \, &v^2 = 2gy
\end{aligned}
$$
Since the particle starts from rest and moves downwards (increasing $y$), the speed $v$ is positive:
$v = \sqrt{2gy}$

**3. Time Taken along the Curve:**
The time $T$ taken for the particle to travel along a curve is the integral of the infinitesimal time elements $dt$. The relationship between distance, speed, and time is $dt = \frac{ds}{v}$, where $ds$ is an infinitesimal arc length along the curve.
The arc length element $ds$ is given by $ds = \sqrt{dx^2 + dy^2}$.
We can write $ds$ in terms of $x$ and $y(x)$: 
$$ds = \sqrt{1 + \left(\frac{dy}{dx}\right)^2} dx = \sqrt{1 + (y')^2} dx.$$
The total time is 
$$T = \int_{(0,0)}^{(x_f, y_f)} dt = \int_{(0,0)}^{(x_f, y_f)} \frac{ds}{v}.$$
Substitute $v = \sqrt{2gy}$:

$$
\boxed{
T = \int_{(0,0)}^{(x_f, y_f)} \frac{\sqrt{1 + (y')^2}}{\sqrt{2gy}} dx
}
$$

**4. Calculus of Variations (Euler-Lagrange Equation):**
The problem of finding the path that minimizes the integral $T$ is a classic problem in the calculus of variations. We need to find the function $y(x)$ that minimizes the functional $\int_{x_1}^{x_2} F(y, y', x) dx$, where the integrand is $F(y, y', x) = \frac{\sqrt{1 + (y')^2}}{\sqrt{2gy}}$.
The Euler-Lagrange equation for a function $y(x)$ that minimizes $\int F(y, y', x) dx$ is:

$$
\boxed{
\frac{\partial F}{\partial y} - \frac{d}{dx}\left(\frac{\partial F}{\partial y'}\right) = 0
}
$$

In our case, the integrand $F$ does not explicitly depend on $x$, i.e., $F = F(y, y') = \frac{1}{\sqrt{2g}} \frac{\sqrt{1 + (y')^2}}{\sqrt{y}}$. For such cases, the Euler-Lagrange equation simplifies to the Beltrami Identity:
$$F - y' \frac{\partial F}{\partial y'} = C$$
where $C$ is a constant.

**5. Applying the Beltrami Identity:**
First, calculate the required partial derivative $\frac{\partial F}{\partial y'}$:
$$
\begin{aligned}
F &= \frac{1}{\sqrt{2g}} \frac{(1 + (y')^2)^{1/2}}{y^{1/2}}\\
\frac{\partial F}{\partial y'} &= \frac{1}{\sqrt{2g}} \frac{1}{y^{1/2}} \frac{1}{2}(1 + (y')^2)^{-1/2} (2y')\\
\frac{\partial F}{\partial y'} &= \frac{1}{\sqrt{2g}} \frac{y'}{\sqrt{y}\sqrt{1 + (y')^2}}
\end{aligned}
$$

Now substitute $F$ and $\frac{\partial F}{\partial y'}$ into the Beltrami Identity:

$$
\frac{1}{\sqrt{2g}} \frac{\sqrt{1 + (y')^2}}{\sqrt{y}} - y' \left(\frac{1}{\sqrt{2g}} \frac{y'}{\sqrt{y}\sqrt{1 + (y')^2}}\right) = C
$$

Factor out common terms $\frac{1}{\sqrt{2g}\sqrt{y}\sqrt{1+(y')^2}}$:
$$
\begin{aligned}
\frac{1}{\sqrt{2g}\sqrt{y}\sqrt{1+(y')^2}} \left( \left(\sqrt{1 + (y')^2}\right)^2 - (y')^2 \right) &= C\\
\frac{1}{\sqrt{2g}\sqrt{y}\sqrt{1+(y')^2}} (1) &= C
\end{aligned}
$$

Let $K = C\sqrt{2g}$ (another constant).
$$\frac{1}{\sqrt{y}\sqrt{1+(y')^2}} = K$$

Square both sides:
$$
\begin{aligned}
\frac{1}{y(1+(y')^2)} &= K^2\\
y(1+(y')^2) &= \frac{1}{K^2}
\end{aligned}
$$

Let $\frac{1}{K^2} = 2a$, where $a$ is a constant with dimensions of length.
$$y(1 + (y')^2) = 2a$$

Rearrange to solve for $y'$:
$$
\begin{aligned}
1 + (y')^2 &= \frac{2a}{y}\\
(y')^2 &= \frac{2a}{y} - 1 = \frac{2a - y}{y}\\
\frac{dy}{dx} &= \sqrt{\frac{2a - y}{y}}
\end{aligned}
$$

We take the positive root, assuming $x$ increases as $y$ increases for the relevant part of the curve.

**6. Solving the Differential Equation:**
Separate variables:
$$dx = \sqrt{\frac{y}{2a - y}} dy$$

To integrate this, we use the trigonometric substitution specific to this form:

Let $y = a(1 - \cos(\theta))$. As $y$ goes from $0$ to $2a$, $\theta$ goes from $0$ to $\pi$.

From $y = a(1 - \cos \theta)$, we get $dy = a \sin(\theta) d\theta$.

Also, $2a - y = 2a - a(1 - \cos \theta) = a(2 - 1 + \cos \theta) = a(1 + \cos \theta)$.

Substitute these into the integral for $dx$:
$$dx = \sqrt{\frac{1 - \cos \theta}{1 + \cos \theta}} \cdot a \sin(\theta) d\theta$$

Using half-angle identities: $1 - \cos \theta = 2\sin^2(\theta/2)$ and $1 + \cos \theta = 2\cos^2(\theta/2)$.

$$\sqrt{\frac{2\sin^2(\theta/2)}{2\cos^2(\theta/2)}} = \sqrt{\tan^2(\theta/2)} = \tan(\theta/2) \quad \text{(assuming $0 \le \theta < \pi$)}.$$

Using the double-angle identity: $\sin \theta = 2 \sin(\theta/2) \cos(\theta/2)$.

$$
\begin{aligned}
dx &= \tan(\theta/2) \cdot a \cdot 2 \sin(\theta/2) \cos(\theta/2) d\theta\\
   &= \frac{\sin(\theta/2)}{\cos(\theta/2)} \cdot a \cdot 2 \sin(\theta/2) \cos(\theta/2) d\theta\\
   &= a \cdot 2 \sin^2(\theta/2) d\theta\\
   &= a(1 - \cos \theta) d\theta
\end{aligned}
$$


Integrate $dx$ with respect to $\theta$:

$$
\begin{aligned}
\int dx &= \int a(1 - \cos \theta) d\theta\\
\implies x &= a(\theta - \sin \theta) + D
\end{aligned}
$$

where $D$ is the integration constant.

We already have the equation for $y$ from our substitution: $y = a(1 - \cos \theta)$

**7. Determining the Integration Constant:**
The curve starts at $(0,0)$. At this point, $x=0$ and $y=0$. Substitute $y=0$ into $y = a(1 - \cos \theta)$:
$$0 = a(1 - \cos \theta) \implies 1 - \cos \theta = 0 \implies \cos \theta = 1.$$
The simplest value for $\theta$ that gives $\cos \theta = 1$ and corresponds to the start of the curve is $\theta = 0$.\\
Substitute $x=0$ and $\theta=0$ into $x = a(\theta - \sin \theta) + D$:

$$
\begin{aligned}
0 &= a(0 - \sin 0) + D\\
0 &= a(0 - 0) + D\\
0 &= D
\end{aligned}
$$

So, the integration constant $D=0$.

**8. The Brachistochrone Equations:**
The parametric equations for the brachistochrone curve starting at $(0,0)$ with the y-axis pointing *downwards* are:
$$
\boxed{
\begin{aligned}
x(\theta) &= a(\theta - \sin \theta)\\
y(\theta) &= -a(1 - \cos \theta)
\end{aligned}
}
$$
These are the parametric equations for a cycloid generated by a circle of radius $a$ rolling along the x-axis. The constant $a$ is determined by the condition that the curve must pass through the second point $(x_f, y_f)$.



### **Connecting two points with different paths**

In the code shell below, we are going to connect two points with different paths. We chose the starting and ending points of the paths as the starting and ending points of a Cycloid traced when its generating circle is rotated from 0 to 180 degrees.  
All paths connect the starting point $(x_{\text{start}}, y_{\text{start}}) = (0,0)$ to the ending point $(x_{\text{end}}, y_{\text{end}}) = (r\pi, -2r)$, as defined in the `plot_brachistochrone_paths` function below.

#### **1. Cycloid Path (The Brachistochrone)**

The brachistochrone curve is an inverted cycloid. The standard parametric equations for a cycloid generated by a circle of radius $r$ rolling along the x-axis, starting at the origin, are:  
$x = r(\theta - \sin(\theta))$  
$y = r(1 - \cos(\theta))$  
where $\theta$ is the angle of rotation of the rolling circle.

For the brachistochrone, we need an *inverted* cycloid. This means the curve dips downwards. The parametric equations become:  
$x = r(\theta - \sin(\theta))$  
$y = -r(1 - \cos(\theta))$

The path starts at $(0,0)$. Let's check this with the equations:  
At $\theta = 0$:  
$x = r(0 - \sin(0)) = r(0 - 0) = 0$  
$y = -r(1 - \cos(0)) = -r(1 - 1) = 0$  
So, the path correctly starts at $(0,0)$.

The brachistochrone path from $(0,0)$ to $(x_{end}, y_{end}) = (r\pi, -2r)$ corresponds to the first half-arch of this inverted cycloid. This occurs when the parameter $\theta$ goes from $0$ to $\pi$. Let's check the endpoint at $\theta = \pi$:  
At $\theta = \pi$:  
$x = r(\pi - \sin(\pi)) = r(\pi - 0) = r\pi$  
$y = -r(1 - \cos(\pi)) = -r(1 - (-1)) = -r(2) = -2r$  
This matches the required endpoint $(r\pi, -2r)$.

**Code Implementation (`calculate_cycloid_path`):**  
* The code directly implements the parametric equations for the inverted cycloid:  
```python
    x_cycloid = r * (theta - np.sin(theta))
    y_cycloid = -r * (1 - np.cos(theta))
```
* The parameter $\theta$ is generated using `np.linspace(0, np.pi, n)`, creating `n` evenly spaced values between $0$ and $\pi$.  
* The $x$ and $y$ coordinates are calculated for each value of $\theta$.  
* These $(x, y)$ pairs are returned as arrays to be plotted.


> *The cycloid path is plotted using its standard parametric equations for an inverted cycloid. The parameter $\theta$ is varied from $0$ to $\pi$ to generate the specific portion of the curve that starts at $(0,0)$ and ends at $(r\pi, -2r)$, which is the correct path for the brachistochrone problem between these two points.*

#### **2. Line Path**

The straight line path is simply the line segment connecting the start point $(x_{\text{start}}, y_{\text{start}}) = (0,0)$ and the end point $(x_{\text{end}}, y_{\text{end}}) = (r\pi, -2r)$.

The equation of a straight line passing through two points $(x_1, y_1)$ and $(x_2, y_2)$ can be found using the slope-intercept form or parametrically. A simple way to generate points along the segment is to linearly interpolate between the coordinates.

Let a parameter $t$ vary from $0$ to $1$. The points on the line segment are given by:  
$x(t) = x_{start} + t(x_{\text{end}} - x_{\text{start}})$  
$y(t) = y_{start} + t(y_{\text{end}} - y_{\text{start}})$

Substituting the start and end points:  
$x(t) = 0 + t(r\pi - 0) = t \cdot r\pi$  
$y(t) = 0 + t(-2r - 0) = t \cdot (-2r)$

As $t$ goes from $0$ to $1$, $x(t)$ goes from $0$ to $r\pi$ and $y(t)$ goes from $0$ to $-2r$.

**Code Implementation (`calculate_line_path`):**  
* The code uses `np.linspace` directly on the x and y coordinates:  
```python
    x_line = np.linspace(x_start, x_end, n)
    y_line = np.linspace(y_start, y_end, n)
```
* `np.linspace(start, end, n)` generates `n` evenly spaced values between `start` and `end`. By calling it separately but with the same number of points `n`, it effectively samples the line segment at `n` points corresponding to $t$ values from $0$ to $1$. The $i$-th $x$ point $x_i$ and the $i$-th $y$ point $y_i$ together form the $i$-th point on the line segment.


> *The line path is generated by calculating `n` linearly spaced points for the x-coordinates between $x_{start}$ and $x_{end}$, and `n` linearly spaced points for the y-coordinates between $y_{start}$ and $y_{end}$. When paired, these $(x_i, y_i)$ points form the straight line segment connecting the two endpoints.*

#### **3. Circular Path**

The requirement for this path is a circular arc tangent to the y-axis at $(0,0)$ and passing through $(x_{\text{end}}, y_{\text{end}})$.

**Mathematical Derivation:**  
* A circle tangent to the y-axis at the origin must have its center on the x-axis. Let the center be $(R, 0)$, where $R$ is the radius.  
* The equation of such a circle is $(x - R)^2 + (y - 0)^2 = R^2$, which simplifies to $(x - R)^2 + y^2 = R^2$.  
* This equation guarantees the circle passes through $(0,0)$: $(0 - R)^2 + 0^2 = R^2$.  
* To be tangent to the y-axis at $(0,0)$, the slope of the tangent line at $(0,0)$ must be undefined (a vertical line). Differentiating the equation implicitly with respect to $x$:  
    $2(x - R) + 2y \frac{dy}{dx} = 0$  
    $\frac{dy}{dx} = \frac{-(x - R)}{y} = \frac{R - x}{y}$  
    At $(0,0)$, the slope is $\frac{R - 0}{0} = \frac{R}{0}$, which is undefined. This confirms vertical tangency at the origin.  
* The circle must pass through the endpoint $(x_{\text{end}}, y_{\text{end}})$. Substituting this into the equation:
    $$
    \begin{aligned}
    & (x_{\text{end}} - R)^2 + y_{\text{end}}^2 = R^2\\
    \implies \, &x_{\text{end}}^2 - 2x_{\text{end}}R + R^2 + y_{\text{end}}^2 = R^2\\
    \implies \, &x_{\text{end}}^2 - 2x_{\text{end}}R + y_{\text{end}}^2 = 0\\
    \implies \, &2x_{\text{end}}R = x_{\text{end}}^2 + y_{\text{end}}^2\\
    \implies \, &R = \frac{x_{\text{end}}^2 + y_{\text{end}}^2}{2x_{\text{end}}}
    \end{aligned}
    $$

    If $x_{\text{end}} \neq 0$, $R = \frac{x_{\text{end}}^2 + y_{\text{end}}^2}{2x_{\text{end}}}$. This is the required radius.

**Code Implementation (`calculate_circle_params_and_patch`):**  
* The code first checks if `x_end <= 0`, returning the start point if true, as a circle tangent to the y-axis at (0,0) generally can't reach points on or behind the y-axis (unless it's the origin).  
* It calculates the radius `circle_radius` using the derived formula: `circle_radius = (x_end**2 + y_end**2) / (2 * x_end)`.  
* It checks if `circle_radius <= 0`, returning the start point if true (robustness check).  
* To generate points, it uses `np.linspace(0, x_end, n)` to get `n` values for the x-coordinates.  
* It then solves the circle equation $(x - R)^2 + y^2 = R^2$ for $y$: $y = \pm \sqrt{R^2 - (x - R)^2}$.  
* The code uses the negative square root: `y = -np.sqrt(np.maximum(0, circle_radius**2 - (x - circle_radius)**2))`. This generates points on the lower half of the circle, which is appropriate since $y_{\text{end}} = -2r$ is negative. `np.maximum(0, ...)` is used inside the square root for numerical stability to avoid taking the square root of a tiny negative number due to floating-point inaccuracies.  
* These $(x, y)$ pairs are returned.


> *The circular path is based on the derived equation for a circle tangent to the y-axis at the origin. The radius $R$ is calculated mathematically to ensure the circle passes through the specific endpoint $(x_{\text{end}}, y_{\text{end}})$. Points are generated by sampling x-coordinates and calculating the corresponding y-coordinates on the lower half of this derived circle.*

#### **4. Parabolic Path ($x = ay^2$)**

The requirement for this path is a parabolic path with a vertical tangent at origin $(0,0)$ and passing through $(x_{\text{end}}, y_{\text{end}})$.

**Mathematical Derivation:**  
* A parabola with a vertical axis of symmetry and vertex at the origin has the form $y = ax^2$. A parabola with a horizontal axis of symmetry and vertex at the origin has the form $x = ay^2$.  
* The form $x = ay^2$ has a horizontal axis of symmetry (the x-axis) and its vertex is at $(0,0)$.  
* It passes through $(0,0)$: $0 = a \cdot 0^2$.  
* Let's check the tangent at $(0,0)$. Differentiate $x = ay^2$ implicitly with respect to $y$: $\frac{dx}{dy} = 2ay$. At $(0,0)$, $y=0$, so $\frac{dx}{dy} = 0$. A $dx/dy$ slope of 0 means the tangent is parallel to the y-axis, i.e., vertical. This matches the requirement.  
* The parabola must pass through the endpoint $(x_{\text{end}}, y_{\text{end}})$. Substituting this into the equation:  
    $x_{\text{end}} = a \cdot y_{\text{end}}^2$  
    Solve for $a$:  
    If $y_{\text{end}} \neq 0$, $a = \frac{x_{\text{end}}}{y_{\text{end}}^2}$. This is the required parameter.

**Code Implementation (`calculate_parabola_path`):**  
* The code first checks if `y_end == 0`, returning the start point if true. This handles the degenerate case where the endpoint is on the x-axis (and not the origin).  
* It calculates the parameter `a` using the derived formula: `a = x_end / (y_end**2)`.  
* It checks if `a <= 0`, returning the start point if true. For $a \le 0$, the parabola would open to the left or be the y-axis, neither of which connects (0,0) to $(r\pi, -2r)$ where $r>0$.  
* To generate points, it uses `np.linspace(y_start, y_end, n)` to get `n` values for the y-coordinates. This is efficient because the equation $x=ay^2$ directly gives $x$ for a given $y$.  
* It then calculates the corresponding $x$ values using the derived `a`: `x = a * y**2`.  
* These $(x, y)$ pairs are returned.


> *The parabolic path uses the form $x = ay^2$, which ensures it starts at the origin and has a vertical tangent there. The parameter $a$ is calculated mathematically to ensure the parabola passes through the specified endpoint $(x_{\text{end}}, y_{\text{end}})$. Points are generated by sampling y-coordinates and calculating the corresponding x-coordinates on this derived parabola.*


In [ ]:
def calculate_cycloid_path(r, n):
    """Calculates points for the inverted cycloid (brachistochrone)."""
    theta = np.linspace(0, np.pi, n)
    x_cycloid = r * (theta - np.sin(theta))
    y_cycloid = -r * (1 - np.cos(theta))
    return x_cycloid, y_cycloid


def calculate_line_path(x_start, y_start, x_end, y_end, n):
    """Calculates points for a straight line path."""
    x_line = np.linspace(x_start, x_end, n)
    y_line = np.linspace(y_start, y_end, n)
    return x_line, y_line


def calculate_circle_params_and_patch(x_start, y_start, x_end, y_end, n):
    """
    Calculates parameters for a circular arc tangent to the y-axis at (0,0)
    and passing through (x_end, y_end), and returns the Arc patch.
    """

    if x_end <= 0:
        return np.array([x_start]), np.array([y_start])

    circle_radius = (x_end**2 + y_end**2) / (2 * x_end)

    if circle_radius <= 0:
        return np.array([x_start]), np.array([y_start])

    x = np.linspace(0, x_end, n)
    y = -np.sqrt(np.maximum(0, circle_radius**2 - (x - circle_radius) ** 2))
    return x, y


def calculate_parabola_path(x_start, y_start, x_end, y_end, n):
    """
    Calculates points for a parabolic path with a vertical tangent at origin (x=ay^2)
    passing through (x_end, y_end).
    """
    if y_end == 0:
        return np.array([x_start]), np.array([y_start])

    a = x_end / (y_end**2)
    if a <= 0:
        return np.array([x_start]), np.array([y_start])

    y = np.linspace(y_start, y_end, n)
    x = a * y**2

    return x, y


# Main Plotting Function


def plot_paths_with_descent_time(r=4, n=400, save_fig=False, filename=None):
    """
    Calculates and plots the brachistochrone (cycloid), circular,
    linear, and parabolic paths between (0,0) and (r*pi, -2*r).
    Parameters:
    r (float): Radius parameter defining the cycloid's size and endpoint.
               The endpoint will be (r*pi, -2*r).
    n (int): Number of points to calculate for each curve's discretization.

    Returns:
    None: Displays the matplotlib plot.
    """
    plt.style.use("dark_background")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_title("Comparison of Paths", fontsize=14, pad=15)
    fig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)

    # Defining start and end points
    x_start, y_start = 0, 0
    x_end = r * np.pi
    y_end = -2 * r  # Endpoint of the inverted cycloid for theta = pi
    x_padding = r * 0.5
    y_padding = r * 0.5
    ax.set_xlim(x_start - x_padding, x_end + x_padding)
    ax.set_ylim(y_end - y_padding, y_start + y_padding)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])

    # Calculating Paths
    x_cycloid, y_cycloid = calculate_cycloid_path(r, n)
    x_line, y_line = calculate_line_path(x_start, y_start, x_end, y_end, n)
    x_circle, y_circle = calculate_circle_params_and_patch(
        x_start, y_start, x_end, y_end, n
    )
    x_parabola, y_parabola = calculate_parabola_path(x_start, y_start, x_end, y_end, n)

    # Plotting the paths
    ax.plot(
        x_cycloid, y_cycloid, color="cyan", lw=2.5, label="Cycloid (Brachistochrone)"
    )
    ax.plot(x_circle, y_circle, color="blue", lw=2, label="Circular Path")
    ax.plot(x_line, y_line, color="red", lw=2, label="Line Path")
    ax.plot(
        x_parabola, y_parabola, color="lime", lw=2, label="Parabola Path ($x=ay^2$)"
    )

    # Start and end points
    ax.plot(x_start, y_start, "o", color="white", markersize=8, label="Start (0, 0)")
    ax.plot(
        x_end,
        y_end,
        "o",
        color="yellow",
        markersize=8,
        label=f"End ({x_end:.2f}, {y_end:.2f})",
    )
    ax.legend(loc="upper right", fontsize=10)

    # Save the figure optionally
    if save_fig:
        save_dir = "IMAGES"
        os.makedirs(save_dir, exist_ok=True)
        if filename is None:
            filename = "Brachistochrone_paths.png"

        filepath = os.path.join(save_dir, filename)
        print(f"Saving figure to {os.path.abspath(filepath)}...")
        fig.savefig(filepath, dpi=200)
        print("Figure saved successfully")
        plt.close(fig)

    else:
        plt.show()


plot_paths_with_descent_time(r=4, n=400, save_fig=True)

### **Calculating the time of descent in each path**

The fundamental principle for calculating the time taken for a particle to travel along a path $C$ under gravity, starting from rest at height $y_0$, is:
$$T = \int_C \frac{ds}{v}$$
where $ds$ is an infinitesimal arc length along the path, and $v$ is the speed of the particle at that point.

For the coordinate system in this specific case, the start is at $(0,0)$ and the y-axis points upwards. The endpoint $(x_{\text{end}}, y_{\text{end}})$ has $y_{\text{end}} < 0$. Starting from rest at $y_0 = 0$, the speed $v$ at any point $(x,y)$ on the path, by conservation of energy ($\frac{1}{2}mv^2 + mgy = \frac{1}{2}mv_0^2 + mgy_0$), is given by:

$$
\begin{aligned}
\frac{1}{2}mv^2 + mgy &= 0 + 0\\ 
v^2 &= -2gy\\
v = \sqrt{-2gy}
\end{aligned}
$$

This requires $y \le 0$ for the speed to be real, which is consistent with the particle moving downwards from $y=0$.

The arc length element $ds$ can be written as 
$$
\begin{aligned}
ds &= \sqrt{dx^2 + dy^2}\\
   &= \sqrt{1 + \left(\frac{dy}{dx}\right)^2} dx \\
   &= \sqrt{1 + (y')^2} dx
\end{aligned}
$$

Substituting $v$ and $ds$ into the time integral, the time taken is:

$$
\boxed{
T = \int_{x_{\text{start}}}^{x_{\text{end}}} \frac{\sqrt{1 + (y')^2}}{\sqrt{-2gy}} dx
}
$$

This is the general integral form that the `func` function in our code represents, where $y=f(x)$ and $y'=fp(x)$.

Now let's look at each path:

#### **1. Cycloid Path (Analytic Formula)**

The cycloid is the known **brachistochrone**, and the time of descent for a particle starting from rest at the cusp of an inverted cycloid is a known analytic result.

For an inverted cycloid starting at $(0,0)$ and ending at $(r\pi, -2r)$ in our code's coordinate system (corresponding to one half-arch of a cycloid generated by a circle of radius $r$), the time of descent $T_{\text{cycloid}}$ is given by:
$$
\boxed{
T_{\text{cycloid}} = \pi \sqrt{\frac{r}{g}}
}
$$

**Derivation:**
Using the parametric equations for the cycloid in the code's coordinate system ($x = r(\theta - \sin \theta)$, $y = -r(1 - \cos \theta)$), we have:
$$ dx = r(1 - \cos \theta) d\theta \, , \quad dy = -r(-\sin \theta) d\theta = r \sin \theta d\theta$$
Therefor, $ds$ becomes

$$
\begin{aligned}
ds &= \sqrt{dx^2 + dy^2}\\
   &= \sqrt{[r(1 - \cos \theta) d\theta]^2 + [r \sin \theta d\theta]^2}\\
   &= r \sqrt{(1 - \cos \theta)^2 + \sin^2 \theta} d\theta \\
   &= r \sqrt{2(1 - \cos \theta)} d\theta\\
   &= r \sqrt{4\sin^2(\theta/2)} d\theta \quad \text{using the identity } 1 - \cos \theta = 2\sin^2(\theta/2)\\
   &= 2r |\sin(\theta/2)| d\theta\\
   &= 2r \sin(\theta/2) d\theta \quad \text{for } 0 \le \theta \le \pi, \sin(\theta/2) \ge 0
\end{aligned}
$$


The speed is 
$$v = \sqrt{-2gy} = \sqrt{-2g[-r(1 - \cos \theta)]} = \sqrt{2gr(1 - \cos \theta)}.$$

Using $1 - \cos \theta = 2\sin^2(\theta/2)$:

$$v = \sqrt{2gr \cdot 2\sin^2(\theta/2)} = \sqrt{4gr \sin^2(\theta/2)} = 2\sqrt{gr} \sin(\theta/2).$$


The time integral becomes:
$$
\begin{aligned}
T &= \int_0^\pi \frac{ds}{v}\\
  &= \int_0^\pi \frac{2r \sin(\theta/2) d\theta}{2\sqrt{gr} \sin(\theta/2)}\\
  &= \int_0^\pi \frac{r}{\sqrt{gr}} d\theta \\
  &= \pi \sqrt{\frac{r}{g}}
\end{aligned}
$$


**Code Implementation (`calculate_time`):**

*The code directly uses this derived analytic formula: `np.pi * np.sqrt(r / g)`. It includes checks to ensure $r > 0$ and $g > 0$.*

#### **2. Line Path (Analytic Formula)**

The path is a straight line from $(0,0)$ to $(x_{\text{end}}, y_{\text{end}})$.

**Derivation:**
For a straight line segment from $(0,0)$ to $(x_{\text{end}}, y_{\text{end}})$, the equation is $y = \frac{y_{\text{end}}}{x_{\text{end}}} x$ (assuming $x_{\text{end}} \neq 0$).

The arc length of the entire segment is simply the distance between the points: 
$$L = \sqrt{x_{\text{end}}^2 + y_{\text{end}}^2}.$$
The speed at any point $x$ is 
$$v(x) = \sqrt{-2g y(x)} = \sqrt{-2g \frac{y_{\text{end}}}{x_{\text{end}}} x}.$$

The time integral is 
$$T = \int_0^{x_{\text{end}}} \frac{ds}{v} = \int_0^{x_{\text{end}}} \frac{\sqrt{1 + (y')^2}}{\sqrt{-2g y}} dx.$$

For a straight line, $y' = \frac{y_{\text{end}}}{x_{\text{end}}}$, which is a constant.

$$\sqrt{1 + (y')^2} = \sqrt{1 + \left(\frac{y_{\text{end}}}{x_{\text{end}}}\right)^2} = \sqrt{\frac{x_{\text{end}}^2 + y_{\text{end}}^2}{x_{\text{end}}^2}} = \frac{\sqrt{x_{\text{end}}^2 + y_{\text{end}}^2}}{|x_{\text{end}}|}.$$

Assuming $x_{\text{end}} > 0$ (as $x_{\text{end}}=r\pi$ with $r>0$ in the main function), this is $\frac{\sqrt{x_{\text{end}}^2 + y_{\text{end}}^2}}{x_{\text{end}}}$. This is a constant factor we can take out of the integral.
$$
\begin{aligned}
T &= \int_0^{x_{\text{end}}} \frac{\frac{\sqrt{x_{\text{end}}^2 + y_{\text{end}}^2}}{x_{\text{end}}}}{\sqrt{-2g \frac{y_{\text{end}}}{x_{\text{end}}} x}} dx\\
  &= \frac{\sqrt{x_{\text{end}}^2 + y_{\text{end}}^2}}{x_{\text{end}}} \int_0^{x_{\text{end}}} \frac{dx}{\sqrt{-2g \frac{y_{\text{end}}}{x_{\text{end}}} x}}\\
  &= \frac{\sqrt{x_{\text{end}}^2 + y_{\text{end}}^2}}{x_{\text{end}}} \frac{1}{\sqrt{-2g \frac{y_{\text{end}}}{x_{\text{end}}}}} \int_0^{x_{\text{end}}} x^{-1/2} dx\\
  &= \frac{\sqrt{x_{\text{end}}^2 + y_{\text{end}}^2}}{\sqrt{x_{\text{end}}}} \frac{1}{\sqrt{-2g y_{\text{end}}}} \cdot 2\sqrt{x_{\text{end}}}\\
  &= 2 \sqrt{\frac{x_{\text{end}}^2 + y_{\text{end}}^2}{-2g y_{\text{end}}}}
\end{aligned}
$$

**Code Implementation (`calculate_time`):**

*The code uses this derived analytic formula: `2 * np.sqrt(numerator / denominator)` where `numerator = x_end**2 + y_end**2` and `denominator = -2 * g * y_end`. It includes checks to ensure $y_{\text{end}} < 0$ (so the denominator is positive) and the denominator is non-zero.*


#### **3. Circular Path (Numerical Integration)**

The time of descent along the circular arc is calculated using numerical integration. The general integral $T = \int_0^{x_{\text{end}}} \frac{\sqrt{1 + (y')^2}}{\sqrt{-2g y}} dx$ is employed.

**Integral Setup:**
The code defines the circular path tangent to the y-axis at $(0,0)$ passing through $(x_{\text{end}}, y_{\text{end}})$ with the equation $(x - R)^2 + y^2 = R^2$, where $R = \frac{x_{\text{end}}^2 + y_{\text{end}}^2}{2x_{\text{end}}}$. Since the path goes downwards ($y \le 0$), the function for $y(x)$ is $y(x) = -\sqrt{R^2 - (x - R)^2}$.
The derivative $y'(x)$ is found by differentiating this expression:
$y'(x) = \frac{dy}{dx} = \frac{x - R}{\sqrt{R^2 - (x - R)^2}}$.

The integrand `func(x, f, fp, g)` in the code calculates $\frac{\sqrt{1 + [fp(x)]^2}}{\sqrt{-2g \cdot f(x)}}$.
For the circle, `f(x)` is the `circle_f(x)` function, which returns $y(x) = -\sqrt{R^2 - (x - R)^2}$.
`fp(x)` is the `circle_fp(x)` function, which returns $y'(x) = \frac{x - R}{\sqrt{R^2 - (x - R)^2}}$.

**Code Implementation (`numerical_integration`):**

* The code takes the circle radius $R$ (passed as `param`) and the x-endpoint `x_{\text{end}}`.
* It defines two helper functions:
    * `circle_f(x)`: Implements $y(x) = -\sqrt{R^2 - (x - R)^2}$. It uses `max(0, ...)` inside the square root for numerical stability near the endpoints where the term might be slightly negative.
    * `circle_fp(x)`: Implements $y'(x) = \frac{x - R}{\sqrt{R^2 - (x - R)^2}}$. It includes a check for the denominator being near zero (at the endpoints $x=0$ and $x=x_{\text{end}}$ where the tangent is vertical or near vertical) and returns $\pm \infty$ accordingly.
* `scipy.integrate.quad(func, 0, x_{\text{end}}, args=(circle_f, circle_fp, g))` is called. This function performs numerical integration of the `func` integrand from $x=0$ to $x=x_{\text{end}}$, passing `circle_f`, `circle_fp`, and `g` as arguments to `func`.


#### **4. Parabola Path (Numerical Integration)**

The time of descent along the parabolic path is also calculated using numerical integration, based on the general integral $T = \int_0^{x_{\text{end}}} \frac{\sqrt{1 + (y')^2}}{\sqrt{-2g y}} dx$.

**Integral Setup:**
The code defines the parabolic path with a vertical tangent at $(0,0)$ passing through $(x_{\text{end}}, y_{\text{end}})$ using the equation $x = ay^2$, where $a = \frac{x_{\text{end}}}{y_{\text{end}}^2}$. Since the path goes downwards and to the right ($x \ge 0, y \le 0$), the function for $y(x)$ is $y(x) = -\sqrt{x/a}$.
The derivative $y'(x)$ is found by differentiating this expression:
$y'(x) = \frac{dy}{dx} = -\frac{1}{2\sqrt{ax}}$.

The integrand `func(x, f, fp, g)` calculates $\frac{\sqrt{1 + [fp(x)]^2}}{\sqrt{-2g \cdot f(x)}}$.
For the parabola, `f(x)` is the `parabola_f(x)` function, which returns $y(x) = -\sqrt{x/a}$.
`fp(x)` is the `parabola_fp(x)` function, which returns $y'(x) = -\frac{1}{2\sqrt{ax}}$.


**Code Implementation (`numerical_integration`):**
* The code takes the parabola parameter $a$ (passed as `param`) and the x-endpoint $x_{\text{end}}$.
* It defines two helper functions:
    * `parabola_f(x)`: Implements $y(x) = -\sqrt{x/a}$. It uses `max(0, ...)` inside the square root for numerical stability for small $x$.
    * `parabola_fp(x)`: Implements $y'(x) = -\frac{1}{2\sqrt{ax}}$. It includes a check for the denominator being near zero at $x=0$ (where the tangent is vertical) and returns $-\infty$.
* `scipy.integrate.quad(func, 0, x_{\text{end}}, args=(parabola_f, parabola_fp, g))` is called to numerically integrate the `func` integrand from $x=0$ to $x=x_{\text{end}}$, using `parabola_f`, `parabola_fp`, and `g`.



In [ ]:
# Gravitational acceleration (m/s²)
G = 10


# Path Calculation Functions
def calculate_path(path_type, x_end, y_end, r=None, n=400):
    """Calculate points for different path types between (0,0) and (x_end, y_end)"""
    x_start, y_start = 0, 0

    if path_type == "cycloid" and r is not None:
        theta = np.linspace(0, np.pi, n)
        x = r * (theta - np.sin(theta))
        y = -r * (1 - np.cos(theta))
        return x, y, r  # Return r as the parameter

    elif path_type == "line":
        x = np.linspace(x_start, x_end, n)
        y = np.linspace(y_start, y_end, n)
        return x, y, None

    elif path_type == "circle":
        if x_end <= 0:
            return None, None, None

        circle_radius = (x_end**2 + y_end**2) / (2 * x_end)
        if circle_radius <= 0:
            return None, None, None

        x = np.linspace(0, x_end, n)
        y = -np.sqrt(np.maximum(0, circle_radius**2 - (x - circle_radius) ** 2))
        return x, y, circle_radius

    elif path_type == "parabola":
        if y_end == 0:
            return np.array([x_start]), np.array([y_start]), None

        a = x_end / (y_end**2)
        if a <= 0:
            return np.array([x_start]), np.array([y_start]), None

        y = np.linspace(y_start, y_end, n)
        x = a * y**2
        return x, y, a

    return None, None, None


# Time Calculation Functions
def calculate_time(path_type, x_end, y_end, param=None, g=G):
    """Calculate descent time for different path types"""

    if path_type == "cycloid":
        r = param
        if r is None:
            return float("inf")
        return np.pi * np.sqrt(r / g) if r > 0 and g > 0 else float("inf")

    elif path_type == "line":
        if y_end >= 0:
            return float("inf")
        numerator = x_end**2 + y_end**2
        denominator = -2 * g * y_end
        return 2 * np.sqrt(numerator / denominator) if denominator > 0 else float("inf")

    elif path_type == "circle":
        circle_radius = param
        if x_end <= 1e-9 or circle_radius is None or circle_radius <= 0:
            return float("inf")

        return numerical_integration(x_end, circle_radius, "circle", g)

    elif path_type == "parabola":
        a = param
        if x_end <= 1e-9 or a is None or a <= 0:
            return float("inf")

        return numerical_integration(x_end, a, "parabola", g)

    return float("inf")


def func(x, f, fp, g=G):
    """Integrand for time calculation: sqrt(1 + (y')^2) / sqrt(-2gy)"""
    y_coord = f(x)
    yp_val = fp(x)

    # Ensure y is negative for speed calculation
    if y_coord > 1e-9:
        return 0

    # Speed v = sqrt(-2gy)
    denominator = np.sqrt(-2 * g * y_coord + 1e-15)
    numerator = np.sqrt(1 + yp_val**2)

    if denominator < 1e-12:
        return np.inf

    return numerator / denominator


def numerical_integration(x_end, param, path_type, g=G):
    """Perform numerical integration for circle and parabola paths"""
    try:
        time = float("inf")

        if path_type == "circle" and param is not None:
            R = param
            cx = R

            def circle_f(x):
                radicand = max(0, R**2 - (x - cx) ** 2)
                return -np.sqrt(radicand)

            def circle_fp(x):
                radicand = R**2 - (x - cx) ** 2
                if radicand <= 1e-12:
                    return np.sign(x - cx) * np.inf
                return (x - cx) / np.sqrt(radicand)

            time, _ = integrate.quad(
                func, 0, x_end, args=(circle_f, circle_fp, g), limit=100
            )

        elif path_type == "parabola" and param is not None:
            a = param

            def parabola_f(x):
                radicand = max(0, x / a)
                return -np.sqrt(radicand)

            def parabola_fp(x):
                radicand = a * x
                if radicand <= 1e-12:
                    return -np.inf
                return -1 / (2 * np.sqrt(radicand))

            time, _ = integrate.quad(
                func, 0, x_end, args=(parabola_f, parabola_fp, g), limit=100
            )

        return float("inf") if np.isnan(time) else time

    except Exception as e:
        print(f"Error during {path_type} integration: {e}")
        return float("inf")


def plot_paths_with_descent_time(r=4, n_points=400, save_fig=False, filename=None):
    """Plot various paths between (0,0) and (r*pi, -2*r) with descent times"""
    x_end = r * np.pi
    y_end = -2 * r

    # Plotting
    plt.style.use("dark_background")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_title("Comparison of Paths & Descent Times", fontsize=16, pad=10)
    fig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
    x_padding, y_padding = r * 0.5, r * 0.5
    ax.set_xlim(-x_padding, x_end + x_padding)
    ax.set_ylim(y_end - y_padding, y_padding)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])

    # Calculating paths and times
    paths = {}
    times = {}
    colors = {"cycloid": "cyan", "line": "red", "circle": "blue", "parabola": "lime"}

    # Calculating all paths
    for path_type in ["cycloid", "line", "circle", "parabola"]:
        x, y, param = calculate_path(path_type, x_end, y_end, r, n_points)
        if x is not None and y is not None and len(x) > 1:
            paths[path_type] = (x, y)
            times[path_type] = calculate_time(path_type, x_end, y_end, param)

    # Plotting each path
    for path_type in paths:
        x, y = paths[path_type]
        time = times[path_type]

        method = "Analytic" if path_type in ["cycloid", "line"] else "Quad wrt x"
        label = f"{path_type.capitalize()}: T = {time:.3f} s ({method})"
        if path_type == "circle":
            label = f"Circle (y-tan): T = {time:.3f} s ({method})"
        elif path_type == "parabola":
            label = f"Parabola (x=ay²): T = {time:.3f} s ({method})"

        ax.plot(x, y, color=colors[path_type], lw=2, label=label)

    # Filling the area below the circular arc
    x_circle, y_circle = paths["circle"]
    if x_circle is not None and y_circle is not None:
        ax.fill_between(
            x_circle, y_circle, y_end - y_padding, color="dimgray", alpha=0.3
        )

    ax.fill_between([-x_padding, 0], 0, y_end - y_padding, color="dimgray", alpha=0.3)

    ax.plot(0, 0, "o", color="white", markersize=8, label="Start (0, 0)")
    ax.plot(
        x_end,
        y_end,
        "o",
        color="yellow",
        markersize=8,
        label=f"End ({x_end:.2f}, {y_end:.2f})",
    )

    ax.legend(loc="upper right", fontsize="small")

    if save_fig:
        save_dir = "IMAGES"
        os.makedirs(save_dir, exist_ok=True)
        if filename is None:
            filename = "path_with_descent_time.png"

        filepath = os.path.join(save_dir, filename)
        print(f"Saving figure to {os.path.abspath(filepath)}...")
        fig.savefig(filepath, dpi=200)
        print("Figure saved successfully")
        plt.close(fig)

    else:
        plt.show()


plot_paths_with_descent_time(r=4, n_points=400, save_fig=True)

### **Visualizing Tangent and Normal Vectors: Realistic Ball Rolling on a Cycloid**

The `CycloidNormalComparisonAnimator` code explains the calculation and visualization of the tangent and normal vectors, and how the ball's position is offset using the normal vector.

This animator is designed to visually demonstrate *why* offsetting the ball along the normal vector is necessary for a realistic simulation of a ball rolling on a curve. It does this by comparing two identical balls moving along the *same* cycloid path at the same rate: one ball is simply centered *on* the path curve, while the other is offset from the path by its radius along the path's normal vector.

#### **Why Offset Along the Normal Vector?**

When a physical ball (a sphere) rolls along a surface, the point of contact between the ball and the surface is *on* the surface. The center of the ball is located one radius distance away from this point of contact. Because the surface is locally flat at the point of contact (on an infinitesimal scale), the line connecting the center of the ball to the point of contact is perpendicular to the surface at that point.

For a curve, the line perpendicular to the curve at a point is defined by the **normal vector** at that point.

Therefore, to make an animated ball *look* like it's rolling *on* a path curve, its center should not be placed directly *on* the curve, but rather offset from the curve point by the ball's radius in the direction of the curve's normal vector.

#### **Calculating the Tangent and Normal Vectors**

These calculations are primarily handled by the `_cycloid_derivatives` and `_get_tangent_normal_vectors` functions.

1.  **Path Point:** 
    The animation uses the parameter $\theta$, which goes from $0$ to $\pi$ as the ball travels along the cycloid. The exact point $(x_p, y_p)$ on the cycloid path for a given $\theta$ is calculated by the `_cycloid_path` function using the standard parametric equations (in the code's y-upwards coordinate system):
    $$
    \begin{aligned}
    x_p(\theta) &= r(\theta - \sin \theta)\\
    y_p(\theta) &= -r(1 - \cos \theta)
    \end{aligned}
    $$

2.  **Tangent Vector Components:** 
    The tangent vector indicates the direction of the curve at a given point. Its components are given by the derivatives of the parametric equations with respect to the parameter $\theta$. The `_cycloid_derivatives` function calculates these:
    $$
    \begin{aligned}
    \frac{dx}{d\theta} &= \frac{d}{d\theta}[r(\theta - \sin \theta)] = r(1 - \cos \theta)\\
    \frac{dy}{d\theta} &= \frac{d}{d\theta}[-r(1 - \cos \theta)] = -r(0 - (-\sin \theta)) = -r \sin \theta
    \end{aligned}
    $$
    So the tangent vector $\vec{T}$ at $\theta$ is proportional to $(r(1 - \cos \theta), -r \sin \theta)$.

3.  **Unit Tangent Vector:** 
    The `_get_tangent_normal_vectors` function takes these components, calculates their magnitude ($\sqrt{(dx/d\theta)^2 + (dy/d\theta)^2}$), and divides the components by the magnitude to get the unit tangent vector $(\hat{t}_x, \hat{t}_y)$. Special handling is included for $\theta=0$ and $\theta=\pi$, where the magnitude is zero (these are cusps or vertices where the tangent direction needs careful definition).

4.  **Normal Vector Components:**
    A vector normal (perpendicular) to a 2D vector $(A, B)$ is $(-B, A)$ or $(B, -A)$. The `_get_tangent_normal_vectors` function calculates the normal vector by taking the unit tangent vector $(\hat{t}_x, \hat{t}_y)$ and rotating it 90 degrees counter-clockwise. The components of the rotated vector are $(-\hat{t}_y, \hat{t}_x)$.
    So the calculated normal vector components are:
    
    $$
    \boxed{
    \begin{aligned}
    n_x &= -\hat{t}_y\\
    n_y &= \hat{t}_x
    \end{aligned}
    }
    $$

5.  **Choosing the Outward Normal:** 
    For a curved path, the normal vector can point in two opposite directions. We need the normal vector that points "outwards" from the curve, so the ball sits on top. For the cycloid segment from $\theta=0$ to $\pi$ in the code's coordinate system (which is concave upwards), the normal should point generally upwards. The method of rotating the unit tangent by 90 degrees counter-clockwise correctly yields this outward-pointing normal for this specific curve segment.

#### **Visualizing Tangent and Normal Vectors**

The `_update_animation` function draws arrows to represent these vectors on both subplots for each frame:

* **Arrow Start Point:** The arrows are drawn starting from a point on the path. For the left subplot (no offset), the arrows start at the center of the ball, which is placed directly on the path point $(x_p, y_p)$. For the right subplot (with normal offset), the arrows start at the path point $(x_p, y_p)$ itself, which is located directly below the center of the ball (since the ball is offset *from* this point).
* **Arrow End Point:** The arrows extend from their start point in the direction of the vector (tangent or normal) scaled by `self.vector_scale`. For a unit vector $(\vec{v}_x, \vec{v}_y)$ and start point $(x_{start}, y_{start})$, the arrow ends at $(x_{start} + \text{vector\_length} \cdot \vec{v}_x, y_{start} + \text{vector\_length} \cdot \vec{v}_y)$, where `vector_length` is `self.ball_radius * 2 * self.vector_scale`.
* **Color and Labels:** The tangent vector is shown in red with a "Tangent" label, and the normal vector is shown in green with a "Normal" label, positioned near the arrowheads.

#### **Calculating the Ball Position**

The `_calculate_ball_positions` function determines the ball's coordinates:

1.  **Ball 1 (Without Normal Offset):** 
    The position of the ball in the left subplot is simply the path point $(x_p, y_p)$ itself. The ball's center is placed directly on the curve.
    `ball1_x = x_p`
    `ball1_y = y_p`

2.  **Ball 2 (With Normal Offset):** 
    The position of the ball in the right subplot is calculated by offsetting the path point $(x_p, y_p)$ by the `self.ball_radius` along the calculated unit normal vector $(n_x, n_y)$:
    `ball2_x = x_p + self.ball_radius * n_x`
    `ball2_y = y_p + self.ball_radius * n_y`
    This places the center of the ball one radius distance away from the path along the perpendicular direction.

#### **Animation Process (`_update_animation`)**

In each frame, the `_update_animation` function:
1.  Calculates the current time based on the frame number and the total time (`self.time_to_bottom`).
2.  Determines the corresponding parameter $\theta$ for the cycloid based on the current time. (For the cycloid, the time taken to reach a point is proportional to $\theta$).
3.  Calls `_calculate_ball_positions` with this $\theta$ to get the updated coordinates for both balls, as well as the current path point, unit tangent, and unit normal vectors.
4.  Updates the `.center` property of the `matplotlib.patches.Circle` objects representing the balls in both subplots.
5.  Removes the old arrows and text labels from the previous frame and creates new `matplotlib.patches.FancyArrow` and `matplotlib.text.Text` objects at the new calculated positions and orientations. This redraws the vectors and labels at the correct place on the curve for the current time step.
6.  Updates the main title of the figure to show the elapsed time.

By showing the ball centered directly on the path versus the ball offset by the normal, and visualizing the tangent and normal vectors, the animation clearly illustrates *why* the normal offset is the physically correct way to simulate a ball rolling on a curved track. When the ball is just centered on the path (left subplot), it visually "dips into" the curve, especially in highly curved sections. When offset by the normal (right subplot), it appears to rest realistically on the curve.

In [ ]:
class CycloidNormalComparisonAnimator:
    """
    Animator that shows two subplots side-by-side comparing a ball rolling down
    a cycloid with and without the normal position adjustment.
    """

    def __init__(
        self,
        r=4,
        ball_radius=0.2,
        g=9.81,
        n_frames=200,
        save_anim=False,
        filename=None,
        fps=25,
        n_path_points=400,
        vector_scale=1.0,
    ):
        """
        Initializes the animator.

        Args:
            r (float): Cycloid radius parameter
            ball_radius (float): Radius of the animated balls
            g (float): Acceleration due to gravity (m/s^2)
            n_frames (int): Total number of frames for the animation
            save_anim (bool): If True, saves the animation instead of showing it
            filename (str | None): Output filename (without extension) if saving
            fps (int): Frames per second for the animation
            n_path_points (int): Number of points used to define the cycloid curve
            vector_scale (float): Scale factor for tangent and normal vectors
        """
        self.r = r
        self.ball_radius = ball_radius
        self.g = g
        self.n_frames = n_frames
        self.save_anim = save_anim
        self.filename = filename
        self.fps = fps
        self.n_path_points = n_path_points
        self.vector_scale = vector_scale
        # Endpoints
        self.x_end = self.r * np.pi
        self.y_end = -2 * self.r

        # Timer
        self.time_to_bottom = np.pi * np.sqrt(r / g)

        # Animation elements
        self.fig = None
        self.axs = None
        self.balls = None
        self.tangent_arrows = None
        self.normal_arrows = None
        self.tangent_labels = None
        self.normal_labels = None
        self.title = None

    def _cycloid_path(self, theta):
        """Parametric equation for the cycloid."""
        x = self.r * (theta - np.sin(theta))
        y = -self.r * (1.0 - np.cos(theta))
        return x, y

    def _cycloid_derivatives(self, theta):
        """Calculate derivatives of the cycloid at theta."""
        dx_dtheta = self.r * (1.0 - np.cos(theta))
        dy_dtheta = -self.r * np.sin(theta)
        return dx_dtheta, dy_dtheta

    def _get_tangent_normal_vectors(self, theta):
        """Calculate tangent and normal vectors at the given theta."""

        x, y = self._cycloid_path(theta)
        dx_dtheta, dy_dtheta = self._cycloid_derivatives(theta)
        tangent_magnitude = np.sqrt(dx_dtheta**2 + dy_dtheta**2)
        if tangent_magnitude < 1e-10:
            # Handle special case at cusps
            if np.isclose(theta, 0):
                tx, ty = 0, 1  # Right-pointing at start
            elif np.isclose(theta, np.pi):
                tx, ty = 1, 0  # Right-pointing at end
            else:
                tx, ty = 1, 0  # Default if something goes wrong
        else:
            tx = dx_dtheta / tangent_magnitude
            ty = dy_dtheta / tangent_magnitude
        # Normal vector (perpendicular to tangent, pointing "outward")
        nx = -ty  # Rotate tangent 90° counter-clockwise
        ny = tx

        return (x, y), (tx, ty), (nx, ny)

    def _calculate_ball_positions(self, theta):
        """Calculate ball positions (with and without normal offset)."""
        point, (tx, ty), (nx, ny) = self._get_tangent_normal_vectors(theta)
        x, y = point

        # Ball without normal adjustment
        ball1_x = x
        ball1_y = y

        # Ball with normal adjustment
        ball2_x = x + self.ball_radius * nx
        ball2_y = y + self.ball_radius * ny

        return (ball1_x, ball1_y), (ball2_x, ball2_y), point, (tx, ty), (nx, ny)

    def _setup_plot(self):
        """Set up the figure with two subplots."""
        plt.style.use("dark_background")

        self.fig, self.axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
        self.fig.subplots_adjust(
            left=0.05, right=0.95, top=0.90, bottom=0.05, wspace=0.1
        )

        titles = ["Without Normal Offset", "With Normal Offset"]
        for i, ax in enumerate(self.axs):
            # Plot the cycloid path
            theta = np.linspace(0, np.pi, self.n_path_points)
            x, y = self._cycloid_path(theta)
            ax.plot(x, y, color="cyan", lw=2, zorder=1)

            x_padding, y_padding = self.r * 0.3, self.r * 0.3
            x_max = self.x_end + x_padding + self.ball_radius * 4 * self.vector_scale
            ax.set_xlim(-x_padding, x_max)
            ax.set_ylim(self.y_end - y_padding, y_padding + self.ball_radius)
            ax.set_aspect("equal", adjustable="box")
            ax.grid(False)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(titles[i], fontsize=14)

            ax.fill_between(
                x, y, self.y_end - y_padding, color="gray", alpha=0.3, zorder=0
            )
            ax.fill_between(
                [-x_padding, 0],
                0,
                self.y_end - y_padding,
                color="gray",
                alpha=0.3,
                zorder=0,
            )

        # Balls, tangent arrows, normal arrows, and text labels
        self.balls = []
        self.tangent_arrows = []
        self.normal_arrows = []
        self.tangent_labels = []
        self.normal_labels = []
        self.arrow_head_width = 0.5
        self.arrow_head_length = 2 * self.arrow_head_width

        for i in range(2):
            if i == 0:
                # No offset
                ball_x, ball_y = self._cycloid_path(0)
            else:
                # With offset
                _, (ball_x, ball_y), _, _, _ = self._calculate_ball_positions(0)

            # Creates ball
            ball = patches.Circle(
                (ball_x, ball_y), self.ball_radius, color="yellow", zorder=3
            )
            self.balls.append(ball)
            self.axs[i].add_patch(ball)

            # Initialize tangent and normal arrows
            tangent_arrow = self.axs[i].arrow(
                0,
                0,
                0,
                0,
                color="red",
                width=0.1,
                head_width=self.arrow_head_width,
                head_length=self.arrow_head_length,
                zorder=2,
            )
            normal_arrow = self.axs[i].arrow(
                0,
                0,
                0,
                0,
                color="green",
                width=0.1,
                head_width=self.arrow_head_width,
                head_length=self.arrow_head_length,
                zorder=2,
            )
            self.tangent_arrows.append(tangent_arrow)
            self.normal_arrows.append(normal_arrow)

            # Initialize text labels
            tangent_label = self.axs[i].text(
                0,
                0,
                "Tangent",
                color="red",
                fontsize=10,
                ha="center",
                va="center",
                zorder=4,
            )
            normal_label = self.axs[i].text(
                0,
                0,
                "Normal",
                color="green",
                fontsize=10,
                ha="center",
                va="center",
                zorder=4,
            )
            self.tangent_labels.append(tangent_label)
            self.normal_labels.append(normal_label)

        self.title = self.fig.suptitle(
            f"Cycloid Ball Animation: With and Without Normal Offset\n"
            f"Time: 0.00 s / {self.time_to_bottom:.2f} s",
            fontsize=16,
            y=0.98,
        )

    def _update_animation(self, frame):
        """Update animation for each frame."""
        # Updates the timer
        t_ratio = frame / self.n_frames
        current_time = t_ratio * self.time_to_bottom * 1.1

        if current_time >= self.time_to_bottom:
            theta = np.pi  # End position
            display_time = self.time_to_bottom
        else:
            # For cycloid, theta is proportional to time: t = sqrt(r/g) * theta
            # theta = t * sqrt(g/r)
            theta = current_time * np.sqrt(self.g / self.r)
            display_time = current_time

        # Ball positions
        (ball1_x, ball1_y), (ball2_x, ball2_y), path_point, (tx, ty), (nx, ny) = (
            self._calculate_ball_positions(theta)
        )
        path_x, path_y = path_point

        artists_to_update = []
        # Update the first subplot (no normal offset)
        if self.balls and len(self.balls) > 0:
            self.balls[0].center = (ball1_x, ball1_y)
            artists_to_update.append(self.balls[0])

        # Update the second subplot (with normal offset)
        if self.balls and len(self.balls) > 1:
            self.balls[1].center = (ball2_x, ball2_y)
            artists_to_update.append(self.balls[1])
        vector_length = self.ball_radius * 2 * self.vector_scale

        # Remove old arrows and create new ones
        if self.tangent_arrows and self.normal_arrows is not None:
            for i, ax in enumerate(self.axs):  # type: ignore
                # Remove old arrows
                if self.tangent_arrows[i]:
                    self.tangent_arrows[i].remove()
                if self.normal_arrows[i]:
                    self.normal_arrows[i].remove()

                if i == 0:  # First subplot (no offset)
                    ball_x, ball_y = ball1_x, ball1_y
                    # Arrow starts from center of ball
                    start_x, start_y = ball_x, ball_y
                else:  # Second subplot (with offset)
                    ball_x, ball_y = ball2_x, ball2_y
                    # Arrow starts from bottom of ball (opposite to normal direction)
                    start_x = ball_x - self.ball_radius * nx
                    start_y = ball_y - self.ball_radius * ny

                # Create new tangent arrow
                self.tangent_arrows[i] = ax.arrow(
                    start_x,
                    start_y,
                    vector_length * tx,
                    vector_length * ty,
                    color="red",
                    width=0.1,
                    head_width=self.arrow_head_width,
                    head_length=self.arrow_head_length,
                    zorder=2,
                )

                # Create new normal arrow
                self.normal_arrows[i] = ax.arrow(
                    start_x,
                    start_y,
                    vector_length * nx,
                    vector_length * ny,
                    color="green",
                    head_width=self.arrow_head_width,
                    head_length=self.arrow_head_length,
                    zorder=2,
                )

                if (
                    self.tangent_labels
                    and self.normal_labels
                    and self.balls is not None
                ):
                    # Position text labels at the end of the vectors
                    tangent_label_x = start_x + vector_length * tx * 1.8
                    tangent_label_y = start_y + vector_length * ty * 1.2
                    self.tangent_labels[i].set_position(
                        (tangent_label_x, tangent_label_y)
                    )

                    normal_label_x = start_x + vector_length * nx * 1.2
                    normal_label_y = start_y + vector_length * ny * 1.4
                    self.normal_labels[i].set_position((normal_label_x, normal_label_y))

                    artists_to_update.extend(
                        [
                            self.balls[i],
                            self.tangent_arrows[i],
                            self.normal_arrows[i],
                            self.tangent_labels[i],
                            self.normal_labels[i],
                        ]
                    )
        if self.title:
            self.title.set_text(
                f"Cycloid Ball Animation: With and Without Normal Offset\n"
                f"Time: {display_time:.2f} s / {self.time_to_bottom:.2f} s"
            )
            artists_to_update.append(self.title)

        return artists_to_update

    def animate(self):
        """Sets up and runs the animation."""
        self._setup_plot()

        if self.fig:
            ani = FuncAnimation(
                self.fig,
                self._update_animation,
                frames=self.n_frames,
                interval=int(1000 / self.fps),
                blit=False,
                repeat=False,
            )

        if self.save_anim:
            if not self.filename:
                self.filename = f"cycloid_normal_comparison_r{self.r}_g{self.g:.2f}.gif"

            if not self.filename.lower().endswith((".gif", ".mp4", ".avi")):
                self.filename += ".gif"

            save_dir = "ANIMATIONS/CYCLOID_COMPARISON"
            os.makedirs(save_dir, exist_ok=True)
            filepath = os.path.join(save_dir, self.filename)
            print(f"Saving animation to {os.path.abspath(filepath)}...")

            try:
                writer = (
                    "pillow" if self.filename.lower().endswith(".gif") else "ffmpeg"
                )
                ani.save(filepath, writer=writer, fps=self.fps, dpi=120)
                print("Animation saved successfully!")
            except Exception as e:
                print(f"Error saving animation: {e}")
                print("Showing animation instead...")
                plt.show()
            finally:
                plt.close(self.fig)
        else:
            plt.show()

        return self.fig, ani


if __name__ == "__main__":
    animator = CycloidNormalComparisonAnimator(
        r=10,
        ball_radius=1,
        g=9.81,
        n_frames=300,
        fps=50,
        vector_scale=2.5,
        save_anim=True,
    )
    fig, ani = animator.animate()

### **Animating the Brachistochrone motion**

To make the ball look like it is actually rolling down the curve, we offset the center of the ball by a distance equal to its radius along its radius. Here, I have explained how the **offset ball position** is calculated for each type of curve using **normal vectors**. The goal is to find the **center of the rolling ball** at each point along the path, such that the **bottom of the ball touches the path**.


#### **Core Concept**

In all cases, the ball rolls along a curve (path). The visible ball must sit *above* the path by a vertical distance equal to the **ball's radius**, but not strictly vertical — rather, the offset must follow the **normal direction** to the curve at that point. This gives a realistic rolling effect.

Mathematically, for a point $(x, y)$ on the path and a unit **normal vector** $\hat{n} = (n_x, n_y)$, the **ball center** $(x_b, y_b)$ is:

$$
(x_b, y_b) = (x, y) + r_{\text{ball}} \cdot \hat{n}
$$

Where $r_{\text{ball}}$ is the ball's radius.



####  **Curve-by-Curve Derivation**

1. **Cycloid**

   - Parametric Equations:

      The cycloid is defined parametrically by:

      $$
      x(\theta) = r(\theta - \sin\theta), \quad y(\theta) = -r(1 - \cos\theta)
      $$

      Where $r$ is the cycloid generating circle’s radius.

   - Tangent Vector:

      Differentiate with respect to $\theta$:

      $$
      \frac{dx}{d\theta} = r(1 - \cos\theta), \quad \frac{dy}{d\theta} = -r\sin\theta
      $$

      So the **tangent vector** is:

      $$
      \vec{T} = (r(1 - \cos\theta), -r\sin\theta)
      $$

   - Normal Vector:

      The **normal vector** is perpendicular to the tangent. A 2D perpendicular vector $(a, b)$ has perpendicular $(b, -a)$ or $(-b, a)$.

      Use:

      $$
      \vec{N} = (r\sin\theta, r(1 - \cos\theta))
      $$

      Normalize it:

      $$
      |\vec{N}| = r\sqrt{\sin^2\theta + (1 - \cos\theta)^2}
      $$

      $$
      \hat{n} = \left(\frac{\sin\theta}{\sqrt{\sin^2\theta + (1 - \cos\theta)^2}}, \frac{1 - \cos\theta}{\sqrt{\sin^2\theta + (1 - \cos\theta)^2}}\right)
      $$

      Then the **ball center position** is:

      $$
      x_b = x(\theta) + r_{\text{ball}} \cdot n_x, \quad y_b = y(\theta) + r_{\text{ball}} \cdot n_y
      $$

   
   *Now, let's compare this to the normal components used in the code for the cycloid:*
   ```python
   nx_comp = np.sin(theta)
   ny_comp = 1.0 - np.cos(theta)
   ```

   *These components **match** the corrected derived normal vector components $(\sin \theta, 1 - \cos \theta)$.*
   *This normal vector $(\sin \theta, 1 - \cos \theta)$ points outwards from the concave-up cycloid segment (for $0 < \theta < \pi$). For instance:*

   - *If $0 < \theta < \pi$, $\sin \theta > 0$ and $1 - \cos \theta > 0$, so the normal points in the (+x, +y) direction (up and right). The cycloid curve goes down and right, so up-right is indeed outwards.*
   - *The code handles the singularities at $\theta=0$ and $\theta=\pi$ separately, where the tangent becomes vertical or horizontal, and the normal becomes horizontal or vertical, respectively, ensuring the normal points correctly outwards at these points as well.*



2. **Straight Line**

   -  Line Equation:

      The line connects start $(0, 0)$ to end $(x_{\text{end}}, y_{\text{end}})$

      Let:

      $$
      m = \frac{y_{\text{end}}}{x_{\text{end}}}
      $$

      So the line is: $y = mx$

   - Tangent Vector:

      $$
      \vec{T} = (1, m)
      $$

   - Normal Vector:

      Perpendicular vector is:

      $$
      \vec{N} = (-m, 1)
      $$

      Normalize:

      $$
      \hat{n} = \left(\frac{-m}{\sqrt{1 + m^2}}, \frac{1}{\sqrt{1 + m^2}}\right)
      $$

      Given a point $(x, y)$ on the line (from linear interpolation), the **ball center** is:

      $$
      x_b = x + r_{\text{ball}} \cdot n_x, \quad y_b = y + r_{\text{ball}} \cdot n_y
      $$

   
   *A normal vector is perpendicular to $\vec{T}$. We have two choices. One choice is $\vec{N} = (-m, 1)$. Another is $(m, -1)$.*
   *The line goes from $(0,0)$ to $(x_{end}, y_{end})$ where $x_{end} > 0$ and $y_{end} < 0$. The slope $m$ is negative. We want the normal vector to point "above" the line.*
   *Consider $\vec{N} = (-m, 1)$. Since $m<0$, $-m > 0$. The components are $(+,+)$, which points upwards and rightwards. This is the correct direction to offset the ball above the line segment.*

   *The code uses normal components proportional to `(-m, 1)`: `nx_comp = -m`, `ny_comp = 1` (where $m$ is calculated as `self.y_end / self.x_end`).*
   *It handles the vertical line case (`x_end` near 0) setting the normal to $(0, 1)$. As noted previously, for a strictly vertical path descending downwards, a horizontal normal $(\pm 1, 0)$ might be more intuitive for offsetting the ball to the side, but given $x_{end}=r\pi > 0$, this case isn't encountered in the main animation.*




3. **Circle**

   - Circle Arc:

      It passes through $(0, 0)$ and $(x_{\text{end}}, y_{\text{end}})$ with a **vertical tangent** at the start. The center is at:

      $$
      (cx, cy) = (R, 0)
      $$

      The arc is:

      $$
      y = -\sqrt{R^2 - (x - R)^2}
      $$

   - Tangent & Normal:

      You don't differentiate here — instead, **geometrically**, the **normal vector at a point on a circle** points from that point **towards the center**.

      So:

      $$
      \vec{N} = (cx - x, cy - y) = (R - x, -y)
      $$

      Normalize:

      $$
      |\vec{N}| = \sqrt{(R - x)^2 + y^2}, \quad \hat{n} = \left(\frac{R - x}{|\vec{N}|}, \frac{-y}{|\vec{N}|}\right)
      $$

      Then:

      $$
      x_b = x + r_{\text{ball}} \cdot n_x, \quad y_b = y + r_{\text{ball}} \cdot n_y
      $$


   *For a circle, the normal vector at any point on the circumference points either towards the center or away from it, along the radius.*
   *Since the path is on the lower part of the circle ($y \le 0$) and the center is at $(R, 0)$ on the x-axis, the normal vector pointing outwards from the curve and towards the region above it (where the ball sits) is the vector pointing from the path point $(x_p, y_p)$ towards the center $(R, 0)$.*


4. **Parabola**

   - Equation:

      Given $x = a y^2$, you compute the position from $y$. So,

      $$
      x(y) = a y^2
      $$

   - Tangent Vector:

      To find slope $\frac{dy}{dx}$, invert the function:

      $$
      y = -\sqrt{\frac{x}{a}} \quad \text{(since path goes downward)}
      \Rightarrow \frac{dy}{dx} = -\frac{1}{2\sqrt{a x}} = \frac{1}{2 a y}
      $$

      Thus, the tangent vector is:

      $$
      \vec{T} = (1, \frac{1}{2 a y})
      $$

   - Normal Vector:

      Use perpendicular vector:

      $$
      \vec{N} = (1, -2 a y)
      $$

      Normalize:

      $$
      |\vec{N}| = \sqrt{1 + (2 a y)^2}, \quad \hat{n} = \left(\frac{1}{|\vec{N}|}, \frac{-2 a y}{|\vec{N}|}\right)
      $$

      Then:

      $$
      x_b = x(y) + r_{\text{ball}} \cdot n_x, \quad y_b = y + r_{\text{ball}} \cdot n_y
      $$

   *The parabola $x=ay^2$ ($a>0$) opens to the right. The path from $(0,0)$ to $(x_{end}, y_{end})$ has $y \le 0$. For $y < 0$, the term $-2ay$ is positive. The vector $(1, -2ay)$ has components $(+,+)$, pointing upwards and rightwards. This is the correct direction to point "outwards" from the curve for the ball to sit on top.*

The `BrachistochroneAnimator` class is designed to visually simulate and compare how long it takes for a ball to slide down different curved paths under gravity, illustrating the famous Brachistochrone problem (finding the path of fastest descent).

Here are the key methods and their roles:

1.  **`__init__(...)`**:
    * Initializes the animator object.
    * Sets up simulation parameters like the cycloid's scale (`r`), ball size (`ball_radius`), gravity (`g`), animation speed (`fps`), and total frames (`n_frames`).
    * Calculates the fixed start point (0,0) and the common end point (`self.x_end`, `self.y_end`) based on the `r` parameter.
    * Initializes placeholders for the plot elements and calculated data.

2.  **`_calculate_path(path_type, ...)`**:
    * A helper function to calculate the series of (x, y) coordinates that define the shape of a specific `path_type` (like "cycloid", "line", "circle", or "parabola") between the start and end points.
    * It uses the mathematical equations for each curve type and calculates the points based on `self.n_path_points`.
    * It also returns any specific parameter (like the cycloid's 'r', circle's radius, or parabola's 'a') that is needed later for time calculation.

3.  **`_calculate_time(path_type, param, ...)`**:
    * Calculates the total time it would take for a ball to slide from the start to the end point along a given `path_type` under gravity `self.g`.
    * For the Cycloid and Straight Line, it uses known analytic formulas for the descent time.
    * For the Circle and Parabola, it calls `_numerical_integration` to numerically compute the time.

4.  **`_numerical_integration(...)`**:
    * A helper function that performs numerical integration (using `scipy.integrate.quad`) to calculate the descent time for paths (Circle, Parabola) where a simple analytic formula isn't directly used.
    * It integrates the general time integral $\int \frac{ds}{v}$, where $ds$ is the arc length and $v$ is the speed derived from energy conservation ($v = \sqrt{-2gy}$ in this coordinate system). It uses internal helper functions to define $y(x)$ and $y'(x)$ for the specific path type.

5.  **`_calculate_ball_position_[path_type](...)` (e.g., `_calculate_ball_position_cycloid`, `_calculate_ball_position_line`, etc.)**:
    * These functions determine the animated position of the ball's *center* for a specific point on a given path type.
    * They find the corresponding point on the path curve and then offset the ball's center from this path point by `self.ball_radius` in the direction of the curve's normal vector (the vector perpendicular to the path at that point). This makes the ball visually appear to be rolling *on* the curve rather than having the curve pass through its center.

6.  **`_update_animation(frame)`**:
    * This is the core function that runs for each frame of the animation.
    * It calculates the current simulation time based on the frame number and the total times calculated for each path.
    * For each path, it determines the ball's position along that path at the current time (using the `t_ratio` and potentially calling the `_calculate_ball_position_...` helpers).
    * It updates the visual position of each ball (`matplotlib.patches.Circle`) on the plot.
    * It updates the text elements like the elapsed time in the title and the legend labels.
    * It returns the list of updated plot elements to the animation framework (`FuncAnimation`).

7.  **`animate()`**:
    * This is the main method you call to run the animation.
    * It sets up the figure and axes using Matplotlib.
    * It calls `_calculate_path` and `_calculate_time` for all defined path types to get the curve shapes and descent times.
    * It plots the static path curves and initializes the ball objects at their starting positions (offset by the normal at the start).
    * It initializes the legend and title.
    * It creates the `matplotlib.animation.FuncAnimation` object, linking the figure and the `_update_animation` function.
    * Finally, it either displays the animation window or saves the animation to a file based on the `save_anim` parameter.



In [ ]:
class BrachistochroneAnimator:
    """
    Animates balls rolling down different paths (cycloid, line, circle, parabola)
    to demonstrate the Brachistochrone problem.

    Attributes:
        r (float): Parameter controlling the size of the cycloid and the endpoint.
        ball_radius (float): Radius of the balls used in the animation.
        g (float): Acceleration due to gravity.
        n_frames (int): Number of frames in the animation.
        save_anim (bool): Whether to save the animation to a file.
        filename (str | None): Filename for the saved animation. Defaults if None.
        fps (int): Frames per second for the animation.
        n_path_points (int): Number of points to calculate for path discretization.
    """

    DEFAULT_G = 9.81  # Standard Gravitational Acceleration (m/s^2)

    def __init__(
        self,
        r=4,
        ball_radius=0.2,
        g=DEFAULT_G,
        n_frames=200,
        save_anim=False,
        filename=None,
        fps=25,
        n_path_points=400,
    ):
        """
        Initializes the BrachistochroneAnimator.

        Args:
            r (float): Cycloid radius parameter. Defines the endpoint.
            ball_radius (float): Radius of the animated balls.
            g (float): Acceleration due to gravity (m/s^2).
            n_frames (int): Total number of frames for the animation.
            save_anim (bool): If True, saves the animation instead of showing it.
            filename (str | None): Output filename (without extension) if saving.
                                   Defaults to 'brachistochrone_r<r>_g<g>.gif'.
            fps (int): Frames per second for the animation playback/saving.
            n_path_points (int): Number of points used to define each path curve.
        """
        self.r = r
        self.ball_radius = ball_radius
        self.g = g
        self.n_frames = n_frames
        self.save_anim = save_anim
        self.filename = filename
        self.fps = fps
        self.n_path_points = n_path_points
        self.x_end = self.r * np.pi
        self.y_end = -2 * self.r

        # Animation objects placeholders
        self.fig = None
        self.ax = None
        self.paths = {}
        self.times = {}
        self.params = {}
        self.path_lines = {}
        self.legend_handles = {}
        self.balls = {}
        self.title = None
        self.leg = None
        self.min_path = ""
        self.max_path = ""
        self.final_ball_center_x = self.x_end
        self.final_ball_center_y = self.y_end + self.ball_radius

        self.colors = {
            "cycloid": "cyan",
            "line": "red",
            "circle": "blue",
            "parabola": "lime",
        }

    def _calculate_path(self, path_type):
        """Calculate points for different path types between (0,0) and (x_end, y_end)"""
        x_start, y_start = 0.0, 0.0

        if path_type == "cycloid":
            theta = np.linspace(0.0, np.pi, self.n_path_points)
            x = self.r * (theta - np.sin(theta))
            y = -self.r * (1.0 - np.cos(theta))
            x[-1], y[-1] = self.x_end, self.y_end
            return x, y, self.r

        elif path_type == "line":
            x = np.linspace(x_start, self.x_end, self.n_path_points)
            y = np.linspace(y_start, self.y_end, self.n_path_points)
            return x, y, None

        elif path_type == "circle":
            if abs(self.x_end) < 1e-9:
                return None, None, None

            circle_radius = (self.x_end**2 + self.y_end**2) / (2 * self.x_end)
            if circle_radius <= 0:
                print(
                    f"Warning: Circle path calculation resulted in non-positive radius ({circle_radius}). Skipping."
                )
                return None, None, None

            x = np.linspace(0, self.x_end, self.n_path_points)
            y = -np.sqrt(np.maximum(0, circle_radius**2 - (x - circle_radius) ** 2))
            x[-1], y[-1] = self.x_end, self.y_end
            return x, y, circle_radius

        elif path_type == "parabola":
            if abs(self.y_end) < 1e-9:
                return (
                    np.array([x_start, self.x_end]),
                    np.array([y_start, self.y_end]),
                    None,
                )
            a = self.x_end / (self.y_end**2)
            if a <= 0:
                print(
                    f"Warning: Parabola path calculation resulted in non-positive 'a' ({a}). Skipping."
                )
                return None, None, None

            y = np.linspace(y_start, self.y_end, self.n_path_points)
            x = a * y**2
            x[-1], y[-1] = self.x_end, self.y_end
            return x, y, a

        return None, None, None

    def _calculate_time(self, path_type, param):
        """Calculate descent time for different path types"""

        if path_type == "cycloid":
            r_cyc = param
            # Standard time, T = pi * sqrt(r / g) for cycloid
            return (
                np.pi * np.sqrt(r_cyc / self.g)
                if r_cyc > 0 and self.g > 0
                else float("inf")
            )

        elif path_type == "line":
            # Time for straight line path from (0,0) to (x_end, y_end)
            if self.y_end >= 0:
                return float("inf")
            distance_sq = self.x_end**2 + self.y_end**2
            numerator = 2 * distance_sq
            denominator = -self.g * self.y_end
            return np.sqrt(numerator / denominator) if denominator > 0 else float("inf")

        elif path_type == "circle":
            circle_radius = param
            if abs(self.x_end) < 1e-9 or circle_radius is None or circle_radius <= 0:
                return float("inf")
            return self._numerical_integration(circle_radius, "circle")

        elif path_type == "parabola":
            a = param
            if abs(self.x_end) < 1e-9 or a is None or a <= 0:
                return float("inf")
            return self._numerical_integration(a, "parabola")

        return float("inf")

    def _integrand_func(self, x, f, fp):
        """Integrand for time calculation: sqrt(1 + (y')^2) / sqrt(-2gy)"""
        y_coord = f(x)
        yp_val = fp(x)

        if y_coord >= -1e-12:
            return 1e15

        denominator_sq = -2 * self.g * y_coord
        if denominator_sq < 1e-15:
            return 1e15
        denominator = np.sqrt(denominator_sq)

        numerator_sq = 1 + yp_val**2
        if math.isinf(numerator_sq):
            return 1e15
        numerator = np.sqrt(numerator_sq)

        if denominator < 1e-12:
            return 1e15

        result = numerator / denominator
        return result if np.isfinite(result) else 1e15

    def _numerical_integration(self, param, path_type):
        """Perform numerical integration for circle and parabola paths"""
        try:
            time = float("inf")

            if path_type == "circle" and param is not None:
                R = param
                cx = R

                def circle_f(x):
                    # y = -sqrt(R^2 - (x-R)^2)
                    radicand = R**2 - (x - cx) ** 2
                    return -np.sqrt(np.maximum(1e-15, radicand))

                def circle_fp(x):
                    # dy/dx = (x-R) / sqrt(R^2 - (x-R)^2)
                    radicand = R**2 - (x - cx) ** 2
                    if radicand <= 1e-12:
                        return np.sign(x - cx) * 1e15
                    return (x - cx) / np.sqrt(radicand)

                time, abserr = integrate.quad(
                    self._integrand_func,
                    1e-9,
                    self.x_end,
                    args=(circle_f, circle_fp),
                    limit=200,
                    epsabs=1e-6,
                    epsrel=1e-6,
                )

            elif path_type == "parabola" and param is not None:
                a = param

                def parabola_f(x):
                    # y = -sqrt(x/a)  (negative root)
                    radicand = x / a
                    return -np.sqrt(np.maximum(1e-15, radicand))

                def parabola_fp(x):
                    # dy/dx = -1 / (2*sqrt(a*x))
                    radicand = a * x
                    if radicand <= 1e-12:
                        return -1e15
                    return -1 / (2 * np.sqrt(radicand))

                time, abserr = integrate.quad(
                    self._integrand_func,
                    1e-9,
                    self.x_end,
                    args=(parabola_f, parabola_fp),
                    limit=200,
                    epsabs=1e-6,
                    epsrel=1e-6,
                )

            return float("inf") if not np.isfinite(time) or time < 0 else time

        except Exception as e:
            print(f"Error during {path_type} numerical integration: {e}")
            if isinstance(e, integrate.IntegrationWarning):
                print("Scipy Quad Warning:", e)
            return float("inf")

    # Ball Position Calculation Functions
    def _calculate_ball_position_cycloid(self, theta):
        """Calculate cycloid path point and ball center offset by radius along normal."""
        x = self.r * (theta - np.sin(theta))
        y = -self.r * (1.0 - np.cos(theta))
        nx_comp = np.sin(theta)
        ny_comp = 1.0 - np.cos(theta)
        magnitude = np.sqrt(nx_comp**2 + ny_comp**2)

        if np.isclose(theta, 0):
            nx, ny = 1, 0
        elif np.isclose(theta, np.pi):
            nx, ny = 0, 1
        else:
            nx = nx_comp / magnitude
            ny = ny_comp / magnitude

        # Ball center position
        x_ball = x + self.ball_radius * nx
        y_ball = y + self.ball_radius * ny

        return x_ball, y_ball

    def _calculate_ball_position_line(self, x_path, y_path):
        """Calculate line path point and ball center offset by radius along normal."""
        if abs(self.x_end) < 1e-10:
            nx, ny = 0, 1
        else:
            m = self.y_end / self.x_end
            denominator = np.sqrt(m**2 + 1)
            nx = -m / denominator
            ny = 1 / denominator

        x_ball = x_path + self.ball_radius * nx
        y_ball = y_path + self.ball_radius * ny

        return x_ball, y_ball

    def _calculate_ball_position_circle(self, x_path):
        """Calculate circle path point and ball center offset by radius along normal."""
        R = self.params["circle"]
        cx = R

        radicand = R**2 - (x_path - cx) ** 2
        if radicand < 0:
            y_path = self.y_end if x_path >= self.x_end else 0
        else:
            y_path = -np.sqrt(radicand)

        center_x, center_y = cx, 0
        dx = center_x - x_path
        dy = center_y - y_path

        magnitude = np.sqrt(dx**2 + dy**2)
        if magnitude < 1e-10:
            nx, ny = 1, 0
        else:
            nx = dx / magnitude
            ny = dy / magnitude

        x_ball = x_path + self.ball_radius * nx
        y_ball = y_path + self.ball_radius * ny

        return x_ball, y_ball

    def _calculate_ball_position_parabola(self, y_path):
        """Calculate parabola path point and ball center offset by radius along normal."""
        a = self.params["parabola"]

        x_path = a * y_path**2

        nx_comp = 1
        ny_comp = -2 * a * y_path
        magnitude = np.sqrt(nx_comp**2 + ny_comp**2)

        if magnitude < 1e-10:
            nx, ny = 1, 0
        else:
            nx = nx_comp / magnitude
            ny = ny_comp / magnitude

        # Ball center position
        x_ball = x_path + self.ball_radius * nx
        y_ball = y_path + self.ball_radius * ny

        return x_ball, y_ball

    def _update_animation(self, frame):
        """Updates the position of each ball and the legend for each frame."""
        sim_duration = max(t for t in self.times.values() if t < float("inf")) * 1.05
        current_time = (frame / self.n_frames) * sim_duration

        artists_to_update = []

        for path_type in self.paths:
            max_path_time = self.times[path_type]

            if max_path_time == float("inf"):
                x_ball, y_ball = self._get_start_ball_pos(path_type)
                self.balls[path_type].center = (x_ball, y_ball)
                label = f"{path_type.capitalize()} (Fail/Inf Time)"
                self.legend_handles[path_type].set_label(label)
                artists_to_update.append(self.balls[path_type])
                continue

            t_ratio = (
                min(current_time / max_path_time, 1.0) if max_path_time > 0 else 0.0
            )
            display_time = t_ratio * max_path_time

            if t_ratio >= 1.0:
                x_ball, y_ball = self.final_ball_center_x, self.final_ball_center_y
            else:
                if path_type == "cycloid":
                    theta = t_ratio * np.pi
                    x_ball, y_ball = self._calculate_ball_position_cycloid(theta)

                elif path_type == "line":
                    x_path = t_ratio * self.x_end
                    y_path = t_ratio * self.y_end
                    x_ball, y_ball = self._calculate_ball_position_line(x_path, y_path)

                elif path_type == "circle":
                    idx = int(t_ratio * (self.n_path_points - 1))
                    x_path = self.paths[path_type][0][idx]
                    x_ball, y_ball = self._calculate_ball_position_circle(x_path)

                elif path_type == "parabola":
                    y_path = t_ratio * self.y_end
                    x_ball, y_ball = self._calculate_ball_position_parabola(y_path)

            self.balls[path_type].center = (x_ball, y_ball)
            artists_to_update.append(self.balls[path_type])

            label_base = path_type.capitalize()
            if path_type == "circle":
                label_base = "Circle (Vert Tan)"
            if path_type == "parabola":
                label_base = "Parabola (x=ay²)"
            label = f"{label_base}: t = {display_time:.3f} s / {max_path_time:.3f} s"
            self.legend_handles[path_type].set_label(label)

        if self.ax is not None:
            self.leg = self.ax.legend(
                handles=self.legend_handles.values(), loc="upper right", fontsize=12
            )
        artists_to_update.append(self.leg)

        title_text = (
            f"Brachistochrone Problem\n"
            f"Fastest: {self.min_path.capitalize()} ({self.times[self.min_path]:.3f} s), "
            f"Slowest: {self.max_path.capitalize()} ({self.times[self.max_path]:.3f} s)\n"
            f"Elapsed time: {current_time:.2f} s"
        )

        if self.title is not None:
            self.title.set_text(title_text)
        artists_to_update.append(self.title)

        return artists_to_update

    def _get_start_ball_pos(self, path_type):
        """Calculate the initial position of the ball center for a given path type."""
        if path_type == "cycloid":
            return self._calculate_ball_position_cycloid(theta=1e-9)
        elif path_type == "line":
            return self._calculate_ball_position_line(x_path=0, y_path=0)
        elif path_type == "circle":
            return self._calculate_ball_position_circle(x_path=1e-9)
        elif path_type == "parabola":
            return self._calculate_ball_position_parabola(y_path=-1e-9)
        else:
            return (0, self.ball_radius)

    def animate(self):
        """Sets up the plot and runs the brachistochrone animation."""

        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(figsize=(12, 9))
        self.fig.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.05)

        x_padding, y_padding = self.r * 0.5, self.r * 0.5
        self.ax.set_xlim(-x_padding, self.x_end + x_padding)
        self.ax.set_ylim(self.y_end - y_padding, y_padding + self.ball_radius)
        self.ax.set_aspect("equal", adjustable="box")
        self.ax.grid(False)
        self.ax.set_xticks([])
        self.ax.set_yticks([])

        path_types_to_run = ["cycloid", "line", "circle", "parabola"]
        for path_type in path_types_to_run:
            x, y, param = self._calculate_path(path_type)
            if x is not None and y is not None and len(x) > 1:
                self.paths[path_type] = (x, y)
                self.params[path_type] = param
                self.times[path_type] = self._calculate_time(path_type, param)
            else:
                print(f"Skipping path type {path_type} due to calculation issues.")
                self.paths[path_type] = (None, None)
                self.params[path_type] = None
                self.times[path_type] = float("inf")

        print(
            "Calculated Times:",
            {
                k: f"{v:.3f}s" if v < float("inf") else "Inf/Error"
                for k, v in self.times.items()
            },
        )

        valid_times = {
            k: v for k, v in self.times.items() if v < float("inf") and v > 0
        }
        if not valid_times:
            print(
                "Error: No valid finite times calculated for any path. Cannot animate."
            )
            plt.close(self.fig)
            return None, None

        self.min_time = min(valid_times.values())
        self.max_time = max(valid_times.values())

        self.min_path = next(k for k, v in valid_times.items() if v == self.min_time)
        self.max_path = next(k for k, v in valid_times.items() if v == self.max_time)

        print(f"Fastest: {self.min_path.capitalize()} ({self.min_time:.3f} s)")
        print(f"Slowest: {self.max_path.capitalize()} ({self.max_time:.3f} s)")

        for path_type in self.paths:
            x, y = self.paths[path_type]
            if x is None or y is None:
                continue

            time_str = (
                f"{self.times[path_type]:.3f} s"
                if self.times[path_type] < float("inf")
                else "Inf"
            )
            label_base = path_type.capitalize()
            if path_type == "circle":
                label_base = "Circle (Vert Tan)"
            if path_type == "parabola":
                label_base = "Parabola (x=ay²)"
            label = f"{label_base}: t = 0.000 s / {time_str}"

            (line,) = self.ax.plot(
                x, y, color=self.colors[path_type], lw=2, label=label
            )
            self.path_lines[path_type] = line
            self.legend_handles[path_type] = line

        for path_type in self.paths:
            if self.paths[path_type][0] is not None:
                start_x, start_y = self._get_start_ball_pos(path_type)
                ball = patches.Circle(
                    (start_x, start_y),
                    self.ball_radius,
                    color=self.colors[path_type],
                    zorder=10,
                )
                self.balls[path_type] = ball
                self.ax.add_patch(ball)

        x_circle, y_circle = self.paths.get("circle", (None, None))
        if x_circle is not None and y_circle is not None:
            self.ax.fill_between(
                x_circle, y_circle, self.ax.get_ylim()[0], color="gray", alpha=0.3
            )

        self.ax.fill_between(
            [self.ax.get_xlim()[0], 0],
            0,
            self.ax.get_ylim()[0],
            color="gray",
            alpha=0.3,
        )

        self.leg = self.ax.legend(
            handles=self.legend_handles.values(), loc="upper right", fontsize=12
        )

        initial_title = (
            f"Brachistochrone Problem\n"
            f"Fastest: {self.min_path.capitalize()} ({self.times[self.min_path]:.3f} s), "
            f"Slowest: {self.max_path.capitalize()} ({self.times[self.max_path]:.3f} s)\n"
            f"Elapsed time: 0.00 s"
        )
        self.title = self.ax.set_title(
            initial_title, fontsize=16, pad=10, fontweight="bold"
        )
        ani = FuncAnimation(
            self.fig,
            self._update_animation,
            frames=self.n_frames,
            interval=int(1000 / self.fps),
            blit=False,
            repeat=False,
        )

        if self.save_anim:
            if not self.filename:
                self.filename = f"brachistochrone_r{self.r}_g{self.g:.2f}.gif"
            if not self.filename.lower().endswith((".gif", ".mp4", ".avi")):
                self.filename += ".gif"  # Default to GIF

            save_dir = "ANIMATIONS/BRACHISTOCHRONE"
            os.makedirs(save_dir, exist_ok=True)
            filepath = os.path.join(save_dir, self.filename)
            print(f"Saving animation to {os.path.abspath(filepath)}...")
            try:
                if self.filename.lower().endswith(".gif"):
                    writer = PillowWriter(fps=self.fps)

                else:
                    writer = FFMpegWriter(
                        fps=self.fps,
                        codec="libx264",
                        bitrate=8000,  # 8 Mbps
                        extra_args=[
                            "-pix_fmt",
                            "yuv420p",
                            "-profile:v",
                            "high",
                            "-level",
                            "4.1",
                            "-movflags",
                            "+faststart",
                        ],
                    )

                ani.save(filepath, writer=writer)
                print("Animation saved successfully!")
            except Exception as e:
                print(f"Error saving animation: {e}")
                print("Ensure you have 'Pillow' installed (`pip install Pillow`).")
                print("Attempting to show animation instead.")
                plt.show()
            finally:
                plt.close(self.fig)
        else:
            plt.show()

        return self.fig, ani


if __name__ == "__main__":
    animator = BrachistochroneAnimator(
        r=10,
        ball_radius=0.5,
        g=9.81,
        n_frames=300,
        # save_anim=True,
        # filename="my_brachistochrone_animation",
        fps=50,
    )

    # Run the animation
    fig_obj, anim_obj = animator.animate()


### **Comparison with lines of different slopes**

The class below extends the base `BrachistochroneAnimator` to focus specifically on comparing the descent time along the cycloid versus several straight lines with **variable slopes**. Instead of comparing all paths to a single fixed endpoint, this animator uses points *along the cycloid* itself as the endpoints for the straight lines.

#### **How Cycloid is Compared to Lines of Variable Slopes**

1.  **Defining Endpoints (`_calculate_endpoints`)**:
    * The animator takes a list of angles (`angles_deg`), typically between 0 and 180 degrees.
    * For each angle $\theta$ (converted to radians, $\theta_{\text{rad}}$), it calculates the corresponding point $(x, y)$ on the cycloid using the cycloid parametric equations:
        $x = r(\theta_{\text{rad}} - \sin \theta_{\text{rad}})$
        $y = -r(1 - \cos \theta_{\text{rad}})$
    * Each of these $(x, y)$ points becomes the **endpoint** for a specific straight line path that starts from the origin (0,0).
    * The endpoint for the full cycloid path is the point calculated when $\theta = \pi$ (180 degrees), which is $(r\pi, -2r)$.
    * The function also calculates the time it takes to reach *each of these intermediate cycloid points* if the ball were traveling *along the cycloid itself*. The time $t$ to reach the point corresponding to angle $\theta$ on the cycloid is given by $t = \theta \sqrt{r/g}$ (for $\theta \le \pi$).

2.  **Calculating Paths (`_calculate_path`)**:
    * It calculates the main cycloid path from $\theta=0$ to $\theta=\pi$.
    * For each angle provided, it calculates a straight line path from (0,0) to the specific endpoint (the point on the cycloid corresponding to that angle) determined in `_calculate_endpoints`.

3.  **Calculating Descent Times (`_calculate_time`)**:
    * For the main cycloid path, it calculates the total time to reach $(r\pi, -2r)$ using the analytic formula $\pi \sqrt{r/g}$.
    * For each straight line, it calculates the time to reach its specific endpoint (a point on the cycloid) using the analytic formula for a line: $T_{\text{line}} = 2 \sqrt{\frac{x_{\text{end}}^2 + y_{\text{end}}^2}{-2gy_{\text{end}}}}$, where $(x_{\text{end}}, y_{\text{end}})$ is the endpoint for that particular line.

4.  **Animation and Comparison (`_update_animation`)**:
    * The animation simulates balls rolling down the main cycloid and each of the variable-slope straight lines.
    * A table is displayed on the plot which, for each line endpoint, compares two times:
        * The time taken by the ball to reach that endpoint along the *straight line*.
        * The time taken by the ball to reach that endpoint along the *cycloid*.

#### **Deriving Cycloid Time to an Intermediate Point**

As mentioned, the time $t$ to reach a point on the cycloid corresponding to the parameter $\theta$ (where $0 \le \theta \le \pi$) is given by:

$$
\boxed{
t = \theta \sqrt{\frac{r}{g}}
}
$$

**Derivation:**
From the derivation of the cycloid descent time in the previous response, the time integral with $\theta$ as the parameter is $T = \int_0^{\theta_f} \sqrt{\frac{r}{g}} d\theta'$, where $\theta_f$ is the value of the parameter at the endpoint.
If we want the time $t$ to reach any intermediate point parameterized by $\theta$ (where $0 \le \theta \le \pi$), we integrate from $0$ to $\theta$:

$$
\begin{aligned}
t &= \int_0^{\theta} \sqrt{\frac{r}{g}} d\theta'\\
  &= \sqrt{\frac{r}{g}} [\theta']_0^{\theta}\\
  &= \theta \sqrt{\frac{r}{g}}
\end{aligned}
$$
This formula is used in the `_calculate_endpoints` function to determine `cycloid_times_to_endpoints` for the line endpoints.

#### **The Lesson: Why the Cycloid Always Wins**

The animation and the table vividly demonstrate the core lesson of the Brachistochrone problem: **The cycloid is the path of fastest descent between any two points (where the starting point is higher), and is faster than *any* straight line connecting the same two points.**

More specifically, in this animator, the lesson is: 
> **For any point on the cycloid (other than the start), the time taken to reach that point along the cycloid is less than the time taken to reach the same point along the straight line connecting the start (0,0) to that point.**

**Why does the cycloid consistently win, even against lines of increasing slopes?**

1.  **Initial Advantage:** The cycloid starts with a vertical tangent at the origin. This means the ball immediately experiences the full force of gravity pulling it directly downwards, resulting in the maximum possible initial acceleration ($g$) and the fastest initial gain in speed. Any straight line from the origin to a point with $x>0$ starts with a finite or zero slope, meaning the initial acceleration along the line is $g \sin\alpha$, where $\alpha$ is the angle the line makes with the horizontal ($\alpha \le 90^\circ$). $\sin\alpha < 1$ unless the line is vertical ($x_{\text{end}}=0$, which is not the case here). The cycloid's vertical start gives it a crucial early speed boost.

2.  **Optimal Speed Profile:** While a steeper line gains speed faster than a gentler one, the cycloid's shape provides an optimal balance. It's steep at the beginning to gain speed quickly, but then curves to a less steep slope. It maintains a higher average speed compared to a straight line, even though the straight line is the shortest distance. The cycloid strategically trades the increased distance of the curved path for a more advantageous velocity profile throughout the descent.

3.  **The Table's Story:** The table directly quantifies this. For each line endpoint, you'll see the time taken along the cycloid to reach that point is always less than the time taken along the corresponding straight line. As the line gets steeper (angles approach 0 degree), the straight line time decreases, but it never quite catches up to the cycloid time to that same point. The fastest straight line would be a purely vertical drop (angle 90 degrees if endpoint was (0, $y_{\text{end}}$)), but that's only possible if the endpoint is directly below the start ($x_{\text{end}}=0$), which is not the general case.

The animation visually reinforces this by showing the ball on the cycloid consistently pulling ahead of the balls on the straight lines, regardless of their slope, reaching each common horizontal level and ultimately each endpoint first. The lesson is a powerful illustration that the path of quickest descent is not the path of shortest distance, highlighting the counter-intuitive nature of the brachistochrone solution.

In [ ]:
class BrachistochroneVariableSlopesAnimator(BrachistochroneAnimator):
    """
    Animates balls rolling down a cycloid and straight lines with endpoints
    defined by cycloid points at user-specified angles to demonstrate the
    Brachistochrone problem with variable slope lines.

    Attributes:
        angles_deg (list[float]): List of angles in degrees defining cycloid endpoints.
        endpoints (dict): Dictionary mapping path types to their endpoints (x_end, y_end).
        cycloid_times_to_endpoints (dict): Times to reach each line endpoint along cycloid.
    """

    def __init__(
        self,
        angles_deg=[90, 135, 180],
        r=4,
        ball_radius=0.2,
        g=9.81,
        n_frames=200,
        save_anim=False,
        filename=None,
        fps=25,
        n_path_points=400,
    ):
        """
        Initializes the BrachistochroneVariableSlopesAnimator.

        Args:
            angles_deg (list[float]): Angles in degrees for cycloid endpoints (0 < angle <= 180).
            r, ball_radius, g, n_frames, save_anim, filename, fps, n_path_points: As in parent.
        """
        super().__init__(
            r=r,
            ball_radius=ball_radius,
            g=g,
            n_frames=n_frames,
            save_anim=save_anim,
            filename=filename,
            fps=fps,
            n_path_points=n_path_points,
        )

        # Validate and store angles
        self.angles_deg = [angle for angle in angles_deg if 0 < angle <= 180]
        if not self.angles_deg:
            print("No valid angles provided. Using default: [90, 135, 180]")
            self.angles_deg = [90, 135, 180]
        self.angles_rad = [np.deg2rad(angle) for angle in self.angles_deg]

        self.endpoints = {}
        self.cycloid_times_to_endpoints = {}
        self._calculate_endpoints()

        self.x_end = self.endpoints["cycloid"][0]
        self.y_end = self.endpoints["cycloid"][1]

        # Set filename
        if self.filename is None and save_anim:
            angles_str = "_".join([str(int(a)) for a in self.angles_deg])
            self.filename = f"brachistochrone_slopes_angles_{angles_str}_r{self.r}_g{self.g:.2f}.gif"

        self.colors = {"cycloid": "cyan"}
        n_lines = len(self.angles_deg)
        if n_lines > 0:
            colormap = cm.get_cmap("Set2")
            line_colors = [colormap(i / n_lines) for i in range(n_lines)]
            for angle, color in zip(self.angles_deg, line_colors):
                self.colors[f"line_{angle}"] = mcolors.to_hex(color)

        self.table = None

    def _calculate_endpoints(self):
        """Calculate endpoints for cycloid and lines, and cycloid times to line endpoints."""
        theta_cycloid = np.pi
        x_cycloid = self.r * (theta_cycloid - np.sin(theta_cycloid))
        y_cycloid = -self.r * (1.0 - np.cos(theta_cycloid))
        self.endpoints["cycloid"] = (x_cycloid, y_cycloid)
        self.cycloid_times_to_endpoints["cycloid"] = np.pi * np.sqrt(self.r / self.g)

        for angle_deg, angle_rad in zip(self.angles_deg, self.angles_rad):
            path_type = f"line_{angle_deg}"
            x_end = self.r * (angle_rad - np.sin(angle_rad))
            y_end = -self.r * (1.0 - np.cos(angle_rad))
            self.endpoints[path_type] = (x_end, y_end)
            time_to_endpoint = np.sqrt(self.r / self.g) * angle_rad
            self.cycloid_times_to_endpoints[path_type] = time_to_endpoint

    def _calculate_path(self, path_type):
        """Calculate paths for cycloid and lines to their respective endpoints."""
        x_start, y_start = 0.0, 0.0

        if path_type == "cycloid":
            return super()._calculate_path(path_type)

        elif path_type.startswith("line_"):
            x_end, y_end = self.endpoints[path_type]
            x = np.linspace(x_start, x_end, self.n_path_points)
            y = np.linspace(y_start, y_end, self.n_path_points)
            return x, y, None

        return None, None, None

    def _calculate_time(self, path_type, param):
        """Calculate descent time for cycloid and lines."""
        if path_type == "cycloid":
            return super()._calculate_time(path_type, param)

        elif path_type.startswith("line_"):
            x_end, y_end = self.endpoints[path_type]
            if y_end >= 0:
                return float("inf")
            distance_sq = x_end**2 + y_end**2
            numerator = 2 * distance_sq
            denominator = -self.g * y_end
            return np.sqrt(numerator / denominator) if denominator > 0 else float("inf")

        return float("inf")

    def _calculate_ball_position_line(self, x_path, y_path, path_type):
        """Calculate ball position for lines with different slopes."""
        x_end, y_end = self.endpoints[path_type]
        if abs(x_end) < 1e-10:
            nx, ny = 1, 0
        else:
            m = y_end / x_end
            denominator = np.sqrt(m**2 + 1)
            nx = -m / denominator
            ny = 1 / denominator

        x_ball = x_path + self.ball_radius * nx
        y_ball = y_path + self.ball_radius * ny
        return x_ball, y_ball

    def _get_start_ball_pos(self, path_type):
        """Calculate initial ball position."""
        if path_type == "cycloid":
            return super()._get_start_ball_pos(path_type)
        elif path_type.startswith("line_"):
            return self._calculate_ball_position_line(0, 0, path_type)
        return (0, self.ball_radius)

    def _update_animation(self, frame):
        """Update ball positions and table for each frame."""
        sim_duration = max(t for t in self.times.values() if t < float("inf")) * 1.05
        current_time = (frame / self.n_frames) * sim_duration

        artists_to_update = []

        for i, path_type in enumerate(self.path_order):
            max_path_time = self.times[path_type]
            x_end, y_end = self.endpoints[path_type]

            if path_type == "cycloid":
                final_ball_center_x, final_ball_center_y = (
                    self._calculate_ball_position_cycloid(np.pi)
                )
            elif path_type == "line_180":
                final_ball_center_x, final_ball_center_y = (
                    self._calculate_ball_position_cycloid(np.pi)
                )
            elif path_type.startswith("line_"):
                final_ball_center_x, final_ball_center_y = (
                    self._calculate_ball_position_line(x_end, y_end, path_type)
                )
            else:
                final_ball_center_x, final_ball_center_y = (
                    x_end,
                    y_end + self.ball_radius,
                )

            if max_path_time == float("inf"):
                x_ball, y_ball = self._get_start_ball_pos(path_type)
                time_str = "Inf / Inf"
                display_time = 0.0
            else:
                t_ratio = (
                    min(current_time / max_path_time, 1.0) if max_path_time > 0 else 0.0
                )
                display_time = t_ratio * max_path_time
                time_str = f"{display_time:.3f} / {max_path_time:.3f}"

                if t_ratio >= 1.0:
                    x_ball, y_ball = final_ball_center_x, final_ball_center_y
                else:
                    if path_type == "cycloid":
                        theta = t_ratio * np.pi
                        x_ball, y_ball = self._calculate_ball_position_cycloid(theta)
                    elif path_type.startswith("line_"):
                        x_path = t_ratio * x_end
                        y_path = t_ratio * y_end
                        x_ball, y_ball = self._calculate_ball_position_line(
                            x_path, y_path, path_type
                        )

            if path_type in self.balls:
                self.balls[path_type].center = (x_ball, y_ball)
                artists_to_update.append(self.balls[path_type])

            if self.table is not None:
                row_idx = i + 1

                if path_type == "cycloid":
                    col_idx_to_update = 1
                else:
                    col_idx_to_update = 0

                try:
                    cell = self.table.get_celld().get((row_idx, col_idx_to_update))
                    if cell:
                        cell.get_text().set_text(time_str)
                except AttributeError:
                    pass

        title_text = (
            f"Brachistochrone Descent: Cycloid vs. Lines of Varying Slopes\n"
            f"Elapsed time: {current_time:.2f} s"
        )
        self.title.set_text(title_text)
        artists_to_update.append(self.title)

        return artists_to_update

    def animate(self):
        """Set up and run the animation."""
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(figsize=(12, 9))
        self.fig.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.05)

        path_types = ["cycloid"] + [f"line_{angle}" for angle in self.angles_deg]
        self.path_order = []
        self.paths = {}
        self.params = {}
        self.times = {}
        self.path_lines = {}
        self.balls = {}

        for path_type in path_types:
            x, y, param = self._calculate_path(path_type)
            if x is not None and y is not None and len(x) > 1:
                self.paths[path_type] = (x, y)
                self.params[path_type] = param
                self.times[path_type] = self._calculate_time(path_type, param)
                self.path_order.append(path_type)
            else:
                print(f"Skipping path type {path_type} due to calculation issues.")

        print(
            "Calculated Times:",
            {
                k.replace("line_", "Line "): f"{v:.3f}s"
                if v < float("inf")
                else "Inf/Error"
                for k, v in self.times.items()
            },
        )
        print(
            "Cycloid Times to Line Endpoints:",
            {
                k.replace("line_", "Line "): f"{v:.3f}s"
                for k, v in self.cycloid_times_to_endpoints.items()
                if k != "cycloid"
            },
        )

        valid_times = {
            k: v for k, v in self.times.items() if v < float("inf") and v > 0
        }
        if not valid_times:
            print(
                "Error: No valid finite times calculated for any path. Cannot animate."
            )
            plt.close(self.fig)
            return None, None

        valid_endpoints = {k: self.endpoints[k] for k in self.path_order}
        if not valid_endpoints:
            print("Error: No valid endpoints found for axis limits.")
            plt.close(self.fig)
            return None, None

        x_max_coord = max(x for x, y in valid_endpoints.values()) * 1.1
        y_min_coord = min(y for x, y in valid_endpoints.values()) * 1.1

        self.ax.set_xlim(-2, x_max_coord)
        self.ax.set_ylim(y_min_coord, 0.5 + self.ball_radius)
        self.ax.set_aspect("equal", adjustable="box")
        self.ax.grid(False)
        self.ax.set_xticks([])
        self.ax.set_yticks([])

        for path_type in self.path_order:
            x, y = self.paths[path_type]
            if x is None or y is None:
                continue

            zorder = 5 if path_type == "cycloid" else 3
            lw = 3 if path_type == "cycloid" else 2
            (line,) = self.ax.plot(
                x, y, color=self.colors[path_type], lw=lw, zorder=zorder
            )
            self.path_lines[path_type] = line

        for path_type in self.path_order:
            if self.paths[path_type][0] is not None:
                start_x, start_y = self._get_start_ball_pos(path_type)
                ball = patches.Circle(
                    (start_x, start_y),
                    self.ball_radius,
                    color=self.colors[path_type],
                    zorder=10,
                )
                self.balls[path_type] = ball
                self.ax.add_patch(ball)

        x_cycloid, y_cycloid = self.paths.get("cycloid", (None, None))
        if x_cycloid is not None and y_cycloid is not None:
            self.ax.fill_between(
                x_cycloid,
                y_cycloid,
                self.ax.get_ylim()[0],
                color="gray",
                alpha=0.3,
                zorder=0,
            )
        self.ax.fill_between(
            [self.ax.get_xlim()[0], 0],
            0,
            self.ax.get_ylim()[0],
            color="gray",
            alpha=0.3,
            zorder=0,
        )

        col_labels = ["Line Time\n(in seconds)", "Cycloid to Pt\n(in seconds)"]
        row_labels = []
        table_data = []
        row_colors = []
        cell_colors = []

        base_text_color = "white"

        for path_type in self.path_order:
            path_color = self.colors[path_type]
            max_time = self.times[path_type]
            time_str = f"0.000 / {max_time:.3f}" if max_time < float("inf") else "Inf"

            if path_type == "cycloid":
                row_label = "Cycloid"
                line_time_str = ""
                cycloid_total_time_str = time_str
                row_labels.append(row_label)
                table_data.append([line_time_str, cycloid_total_time_str])
                row_colors.append(path_color)
                cell_colors.append([base_text_color, path_color])
            elif path_type.startswith("line_"):
                angle_deg = float(path_type.split("_")[1])
                row_label = f"Line {angle_deg:.0f}°"
                cycloid_time_to_pt = self.cycloid_times_to_endpoints[path_type]
                cycloid_to_pt_str = f"{cycloid_time_to_pt:.3f}"
                row_labels.append(row_label)
                table_data.append([time_str, cycloid_to_pt_str])
                row_colors.append(path_color)
                cell_colors.append([path_color, path_color])

        self.table = self.ax.table(
            cellText=table_data,
            colLabels=col_labels,
            rowLabels=row_labels,
            rowColours=row_colors,
            colColours=["lightgray"] * len(col_labels),
            cellLoc="center",
            rowLoc="center",
            colWidths=[0.15, 0.15],
            loc="upper right",
        )

        self.table.auto_set_font_size(False)
        self.table.set_fontsize(10)
        self.table.scale(1, 2.4)

        for key, cell in self.table.get_celld().items():
            row, col = key
            cell.set_facecolor("black")
            cell.set_edgecolor("gray")
            if row > 0 and col > -1:
                try:
                    text_color = cell_colors[row - 1][col]
                    cell.get_text().set_color(text_color)
                except IndexError:
                    cell.get_text().set_color(base_text_color)
            elif row == 0:
                cell.get_text().set_color("black")
                cell.set_facecolor("lightgray")
                cell.set_text_props(weight="bold")
            elif col == -1 and row > 0:
                cell.get_text().set_color(row_colors[row - 1])
                cell.set_text_props(weight="bold")

        initial_title = (
            "Brachistochrone Descent: Cycloid vs. Lines of Varying Slopes\n"
            "Elapsed time: 0.00 s"
        )
        self.title = self.ax.set_title(
            initial_title, fontsize=18, pad=20, fontweight="bold"
        )

        ani = FuncAnimation(
            self.fig,
            self._update_animation,
            frames=self.n_frames,
            interval=int(1000 / self.fps),
            blit=False,
            repeat=False,
        )

        if self.save_anim:
            save_dir = "ANIMATIONS/BRACHISTOCHRONE"
            os.makedirs(save_dir, exist_ok=True)
            if self.filename is None:
                angles_str = "_".join([str(int(a)) for a in self.angles_deg])
                self.filename = f"brachistochrone_slopes_angles_{angles_str}_r{self.r}_g{self.g:.2f}_table.gif"
            filepath = os.path.join(save_dir, self.filename)
            print(f"Saving animation to {os.path.abspath(filepath)}...")
            try:
                if self.filename.lower().endswith(".gif"):
                    writer = PillowWriter(fps=self.fps)
                else:
                    writer = FFMpegWriter(
                        fps=self.fps,
                        codec="libx264",
                        bitrate=8000,  # 8 Mbps
                        extra_args=[
                            "-pix_fmt",
                            "yuv420p",
                            "-profile:v",
                            "high",
                            "-level",
                            "4.1",
                            "-movflags",
                            "+faststart",
                        ],
                    )
                ani.save(filepath, writer=writer)
                print("Animation saved successfully!")
            except Exception as e:
                print(f"Error saving animation: {e}")
                print("Displaying instead.")
                plt.show()
            finally:
                plt.close(self.fig)
        else:
            plt.show()

        return self.fig, ani


if __name__ == "__main__":
    print(
        "Enter angles in degrees for straight lines (0 < angle <= 180, comma-separated)"
    )
    print("Example: 90,135,180")
    user_input = input("Angles (or press Enter for default [90, 135, 180): ")
    if user_input.strip():
        try:
            angles = [float(angle.strip()) for angle in user_input.split(",")]
            # Validate angles
            angles = [angle for angle in angles if 0 < angle <= 180]
            if not angles:
                raise ValueError("No valid angles in range (0, 180]")
        except ValueError as e:
            print(f"Invalid input: {e}. Using default angles: [90, 135, 180]")
            angles = [90, 135, 180]
    else:
        angles = [90, 135, 180]

    animator = BrachistochroneVariableSlopesAnimator(
        angles_deg=angles,
        r=10,
        ball_radius=0.4,
        g=9.81,
        n_frames=300,
        save_anim=True,
        filename="trial.mp4",
        fps=60,
    )

    fig_obj, anim_obj = animator.animate()


## Isochronous Property of Cycloid

Let's now discuss the tautochrone property of the cycloid and derive why particles starting from different points reach the bottom simultaneously.

The term "tautochrone" comes from Greek words meaning "same time". A tautochrone curve is one where the time taken for an object to slide along it to the lowest point is independent of its starting position on the curve, assuming it starts from rest. The inverted cycloid is the most famous tautochrone curve under uniform gravity.

The `TautochroneAnimator` class below specifically demonstrates this by animating multiple balls starting at different points (defined by `initial_angles_deg`) along a single cycloid, showing that they all arrive at the lowest point at the same time.

### **Deriving the Isochronous Property of the Cycloid**

We start with the parametric equations for the inverted cycloid, matching the coordinate system used in your code (origin at the left cusp, y-axis upwards, lowest point at $(r\pi, -2r)$):

$$
\begin{aligned}
x(\theta) &= r(\theta - \sin \theta)\\
y(\theta) &= -r(1 - \cos \theta)
\end{aligned}
$$

where $\theta$ is the parameter, typically ranging from $0$ to $2\pi$ for one full arch. The lowest point is at $\theta = \pi$.

Consider a particle starting from rest at a point $(x_0, y_0)$ on the cycloid, corresponding to the parameter $\theta_0$. By conservation of energy, the speed $v$ at any point $(x, y)$ on the cycloid (corresponding to parameter $\theta$) is given by:

$$
\begin{aligned}
\frac{1}{2}mv^2 + mgy &= mgy_0\\
v^2 &= 2g(y_0 - y)\\
v &= \sqrt{2g(y_0 - y)}
\end{aligned}
$$

Substituting the parametric equations for $y$:

$$
\begin{aligned}
y_0 - y &= [-r(1 - \cos \theta_0)] - [-r(1 - \cos \theta)]\\
        &= r(\cos \theta_0 - \cos \theta)
\end{aligned}
$$
So,
$$v = \sqrt{2gr(\cos \theta_0 - \cos \theta)}$$

The time $T$ taken to slide from the starting point $\theta_0$ to the lowest point $\theta = \pi$ is given by the integral $T = \int \frac{ds}{v}$. The arc length element $ds$ for the cycloid was derived previously as $ds = 2r |\sin(\theta/2)| d\theta$. For the path from $\theta_0$ down to $\pi$, as $\theta$ varies from $\theta_0$ to $\pi$, $\theta/2$ varies from $\theta_0/2$ to $\pi/2$. For a starting point $\theta_0$ between $0$ and $2\pi$, the ball rolls towards $\theta=\pi$. The direction of travel might mean $\theta$ is increasing or decreasing. Let's use the absolute value of $d\theta$ or handle the integral limits appropriately. The time integral is essentially 
$$\int_{\text{start}}^{\text{end}} \frac{|ds/d\theta|}{v} |d\theta|.$$

Substituting the values of the arc length $ds$ and the limits properly we get, 

$$
\boxed{
T = \int_{\theta_0}^{\pi} \frac{2r |\sin(\theta/2)|}{\sqrt{2gr(\cos \theta_0 - \cos \theta)}} d\theta
}
$$
Assuming the particle starts and rolls down towards $\theta=\pi$, the range of $\theta$ will be within $[0, 2\pi]$. For $0 \le \theta \le 2\pi$, $0 \le \theta/2 \le \pi$, so $\sin(\theta/2) \ge 0$. Thus $|\sin(\theta/2)| = \sin(\theta/2)$. Therefore, 

$$
\begin{aligned}
T &= \int_{\theta_0}^{\pi} \frac{2r \sin(\theta/2)}{\sqrt{2gr(\cos \theta_0 - \cos \theta)}} d\theta\\
  &= \sqrt{\frac{2r}{g}} \int_{\theta_0}^{\pi} \frac{\sin(\theta/2)}{\sqrt{\cos \theta_0 - \cos \theta}} d\theta
\end{aligned}
$$

First, express the term under the square root using the half-angle identity $\cos \alpha = 2\cos^2(\alpha/2) - 1$:
$$\cos \theta_0 - \cos \theta = (2\cos^2(\theta_0/2) - 1) - (2\cos^2(\theta/2) - 1) = 2(\cos^2(\theta_0/2) - \cos^2(\theta/2))$$
Substitute this into the integral:
$$
\begin{aligned}
T &= \sqrt{\frac{2r}{g}} \int_{\theta_0}^{\pi} \frac{\sin(\theta/2)}{\sqrt{2(\cos^2(\theta_0/2) - \cos^2(\theta/2))}} d\theta\\
 &= \sqrt{\frac{r}{g}} \int_{\theta_0}^{\pi} \frac{\sin(\theta/2)}{\sqrt{\cos^2(\theta_0/2) - \cos^2(\theta/2)}} d\theta
\end{aligned}
$$

Now, let's apply the substitution:

Let $\cos(\theta/2) = \cos(\theta_0/2) \cos(\phi/2)$.

We need to change the limits of integration from $\theta$ to $\phi$:
* When $\theta = \theta_0$: $\cos(\theta_0/2) = \cos(\theta_0/2) \cos(\phi_0/2)$. Assuming $\theta_0 \neq \pi$, $\cos(\theta_0/2) \neq 0$. Thus, $\cos(\phi_0/2) = 1$, which implies $\phi_0/2 = 0$, so $\phi_0 = 0$.
* When $\theta = \pi$: $\cos(\pi/2) = 0$. So $0 = \cos(\theta_0/2) \cos(\phi_\pi/2)$. Since $\cos(\theta_0/2) \neq 0$, we need $\cos(\phi_\pi/2) = 0$. The smallest positive value for $\phi_\pi/2$ is $\pi/2$, so $\phi_\pi = \pi$.

The limits of integration for $\phi$ are from $0$ to $\pi$.

Next, we need to express the differential $d\theta$ in terms of $d\phi$. Differentiate the substitution equation with respect to $\phi$:

$$
\begin{aligned}
\frac{d}{d\phi}(\cos(\theta/2)) &= \frac{d}{d\phi}(\cos(\theta_0/2) \cos(\phi/2))\\
\implies \, \sin(\theta/2)\, \frac{d\theta}{d\phi} &= \cos(\theta_0/2) \sin(\phi/2)\\
\implies \, \sin(\theta/2)\, d\theta &= \cos(\theta_0/2) \sin(\phi/2)\, d\phi
\end{aligned}
$$

Now, express the term under the square root in terms of $\phi$:
$$
\begin{aligned}
\sqrt{\cos^2(\theta_0/2) - \cos^2(\theta/2)} &= \sqrt{\cos^2(\theta_0/2) - (\cos(\theta_0/2) \cos(\phi/2))^2}\\
&= \sqrt{\cos^2(\theta_0/2)(1 - \cos^2(\phi/2))}\\ 
&= \sqrt{\cos^2(\theta_0/2) \sin^2(\phi/2)}
\end{aligned}
$$

Since $0 \le \theta_0 < \pi$, $\cos(\theta_0/2) \ge 0$. As $\theta$ goes from $\theta_0$ to $\pi$, $\theta/2$ goes from $\theta_0/2$ to $\pi/2$. $\cos(\theta/2)$ decreases from $\cos(\theta_0/2)$ to $0$. For $\cos(\theta/2) = \cos(\theta_0/2)\cos(\phi/2)$ to hold, as $\cos(\theta/2)$ decreases and $\cos(\theta_0/2)$ is fixed and positive, $\cos(\phi/2)$ must decrease from 1 to 0. This means $\phi/2$ increases from $0$ to $\pi/2$, so $\phi$ increases from $0$ to $\pi$. For $0 \le \phi \le \pi$, $\phi/2$ is in $[0, \pi/2]$, so $\sin(\phi/2) \ge 0$.

Therefore, $\sqrt{\cos^2(\theta_0/2) \sin^2(\phi/2)} = |\cos(\theta_0/2) \sin(\phi/2)| = \cos(\theta_0/2) \sin(\phi/2)$.

Substitute the denominator and $d\theta$ into the integral for $T$:
$$
\begin{aligned}
T &= \sqrt{\frac{r}{g}} \int_{0}^{\pi} \frac{1}{\cos(\theta_0/2) \sin(\phi/2)} \cdot \, \cos(\theta_0/2) \sin(\phi/2) d\phi \\
  &= \sqrt{\frac{r}{g}} \int_{0}^{\pi} d\phi\\
  &= \pi \sqrt{\frac{r}{g}}
\end{aligned}
$$

Notice how the terms dependent on $\theta$ and $\theta_0$ got canceled out.

### **The Expression for the Descent Time**
The time taken for a particle to slide from any starting point on an inverted cycloid to the lowest point under uniform gravity is:
$$
\boxed{
T = \pi \sqrt{\frac{r}{g}}
}
$$
where $r$ is the radius of the generating circle of the cycloid and $g$ is the acceleration due to gravity.

This time is a constant value determined only by the shape of the cycloid (via $r$) and the strength of gravity ($g$). It does **not** depend on the initial height or starting point of the particle on the cycloid.

### **How the Code Demonstrates This**

* The `TautochroneAnimator` takes a list of `initial_angles_deg`.
* For each initial angle $\theta_0$, it uses the relationship $\cos(\theta/2) = \cos(\theta_0/2) \cos(\omega t)$, derived from the equation of motion where $\omega = \sqrt{g/(4r)}$, to calculate the current position parameter $\theta$ at any given time $t$. This is done in `_get_theta_at_time`.
* It then uses the `_get_ball_position` method to find the ($x$, $y$) coordinates corresponding to this $\theta$, offset by the ball radius along the normal.
* The `_update` function animates multiple balls, each starting from a different `theta0`, by calculating their positions over time using the derived motion equation.
* The animation shows all the balls, regardless of their starting height on the cycloid, reaching the lowest point (`theta = pi`) simultaneously, exactly at the time calculated by `self.time_to_bottom = pi * sqrt(r/g)`.

### T**he Lesson We Learn**

The fundamental lesson from the tautochrone property of the cycloid is that:

> **For an object sliding without any friction under uniform gravity on an inverted cycloidal path, the time it takes to reach the lowest point is the same regardless of where it started on the path.**

This is counter-intuitive because particles starting higher have to travel a longer distance, but they gain speed faster due to falling further vertically. The unique shape of the cycloid precisely balances the increased distance against the increased speed, resulting in a constant descent time to the bottom. This property has practical applications, most notably in the design of pendulum clocks (Huygens designed pendulum clocks where the pendulum bob swings along a cycloidal arc to make the period independent of the amplitude of swing, thus making the clock more accurate even as the swing amplitude decreases).

In [ ]:
class TautochroneAnimator:
    """
    Animates the tautochrone problem, showing particles sliding down
    a cycloidal path from different starting points. Demonstrates
    that they reach the bottom simultaneously.
    """

    def __init__(
        self,
        initial_angles_deg,
        r=1.0,
        ball_radius=0.1,
        g=9.81,
        fps=30,
        save_anim=False,
        filename=None,
    ):
        """
        Initializes the Tautochrone Animator.

        Parameters:
        -----------
        initial_angles_deg : float or list[float]
            Starting angle(s) in degrees along the cycloid (0 <= angle <= 360).
            Angle 0 is the left cusp, 180 is the lowest point, 360 is the right cusp.
            If a float, animates one ball. If a list, animates multiple balls.
        r : float
            Radius of the generating circle for the cycloid (default: 1.0).
        ball_radius : float
            Radius of the ball in data units (default: 0.1).
        g : float
            Acceleration due to gravity (default: 9.81). Affects the time scale.
        fps : int
            Frames per second for the animation (default: 30).
        save_anim : bool
            Whether to save the animation to a file (default: False).
        filename : str, optional
            Name of the file to save the animation. If None and save_anim is True,
            a default name is generated.
        """
        self.r = r
        self.g = g
        self.fps = fps
        self.save_anim = save_anim
        self.filename = filename
        self.ball_radius = ball_radius

        if isinstance(initial_angles_deg, (int, float)):
            self.initial_angles_deg = [float(initial_angles_deg)]
        elif isinstance(initial_angles_deg, list):
            self.initial_angles_deg = [float(angle) for angle in initial_angles_deg]
        else:
            raise TypeError("initial_angles_deg must be a number or a list of numbers.")

        if not all(0 <= angle <= 360 for angle in self.initial_angles_deg):
            raise ValueError("Initial angles must be between 0 and 360 degrees.")

        self.theta0_list = [np.radians(angle) for angle in self.initial_angles_deg]
        self.num_balls = len(self.theta0_list)

        # Timing
        self.omega = np.sqrt(self.g / (4 * self.r))
        self.time_to_bottom = np.pi / (2 * self.omega)  # Time = pi * sqrt(r/g)

        self.duration = self.time_to_bottom * 1.2
        self.num_frames = int(self.duration * self.fps)

        self.fig = None
        self.ax = None
        self.balls = []
        self.ball_colors = []

    def _cycloid_path(self, theta):
        """Parametric equation for the inverted cycloid path."""
        x = self.r * (theta - np.sin(theta))
        y = -self.r * (1 - np.cos(theta))
        return x, y

    def _get_theta_at_time(self, theta0, t):
        """Calculates the angular position theta(t) of a ball."""
        if t >= self.time_to_bottom:
            return np.pi
        elif t <= 0:
            return theta0
        else:
            omega = self.omega
            cos_val = np.cos(omega * t)
            if theta0 < np.pi:
                arg = np.cos(theta0 / 2.0) * cos_val
                arg = np.clip(arg, -1.0, 1.0)
                return 2.0 * np.arccos(arg)
            else:
                k = -np.cos(theta0 / 2.0)
                arg = k * cos_val
                arg = np.clip(arg, -1.0, 1.0)
                return 2.0 * np.pi - 2.0 * np.arccos(arg)

    def _get_ball_position(self, theta: float) -> tuple[float, float]:
        """
        Calculates the center position for the ball so it appears
        to rest ON the cycloid curve rather than being centered at the curve.
        """
        x_curve, y_curve = self._cycloid_path(theta)
        if np.isclose(theta, 0) or np.isclose(theta, 2 * np.pi):
            nx = 1 if theta < np.pi else -1
            ny = 0
        elif np.isclose(theta, np.pi):
            nx = 0
            ny = 1
        else:
            nx = self.r * np.sin(theta)
            ny = self.r * (1 - np.cos(theta))

            norm = np.sqrt(nx**2 + ny**2)
            if norm > 1e-9:
                nx /= norm
                ny /= norm

        x_center = x_curve + self.ball_radius * nx
        y_center = y_curve + self.ball_radius * ny

        return x_center, y_center

    def _setup_plot(self):
        """Sets up the matplotlib figure and axes."""
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(figsize=(12, 6))

        self.ax.set_title(
            f"Tautochrone Curve (Cycloid)\nTime to bottom: 0.00s / {self.time_to_bottom:.2f}s",
            fontsize=16,
            pad=10,
        )
        self.fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05)

        x_padding = self.r * 0.2
        y_padding = self.r * 0.2
        self.ax.set_xlim(-x_padding, 2 * np.pi * self.r + x_padding)
        self.ax.set_ylim(-2 * self.r - y_padding, y_padding + self.ball_radius * 2)
        self.ax.set_aspect("equal", adjustable="box")
        self.ax.set_xticks([])
        self.ax.set_yticks([])

        theta_path = np.linspace(0, 2 * np.pi, 400)
        x_path, y_path = self._cycloid_path(theta_path)
        self.ax.plot(x_path, y_path, color="gray", lw=2)
        self.ax.fill_between(
            x_path, y_path, -2 * self.r - y_padding, color="dimgray", alpha=0.5
        )

        color_cycle = itertools.cycle(plt.cm.tab10.colors)  # type: ignore
        for i in range(self.num_balls):
            color = next(color_cycle)
            self.ball_colors.append(color)
            theta0 = self.theta0_list[i]
            x0, y0 = self._get_ball_position(theta0)

            ball = patches.Circle(
                (x0, y0),
                radius=self.ball_radius,
                color=color,
                zorder=5,
                label=f"Ball {i + 1}: {np.degrees(theta0):.0f}$^\\circ$",
            )
            self.balls.append(ball)
            self.ax.add_patch(ball)

        cycloid_eq = (
            "Inverted Cycloid:\n$x = r\\,(\\theta - \\sin\\theta)$, "
            + "$y = -r\\,(1 - \\cos\\theta)$ ; $(0 \\leq \\theta \\leq 2\\pi)$"
        )

        self.equation_text = self.ax.text(
            0.5,
            0.95,
            cycloid_eq,
            transform=self.ax.transAxes,
            ha="center",
            va="top",
            fontsize=12,
            color="white",
            bbox=dict(
                facecolor="black", alpha=0.8, edgecolor="gray", boxstyle="round,pad=0.3"
            ),
        )

        self.ax.legend(bbox_to_anchor=(0.91, 0.97), fontsize=8, ncols=2)

    def _update(self, frame):
        """Updates the animation elements for each frame."""
        t = frame / self.fps
        artists_to_update = []
        current_time = min(t, self.time_to_bottom)
        if self.ax is not None:
            self.ax.set_title(
                f"Tautochrone Curve (Cycloid)\n"
                f"Time to bottom: {current_time:.2f}s / {self.time_to_bottom:.2f}s",
                fontsize=16,
                pad=10,
            )

        for i in range(self.num_balls):
            theta0 = self.theta0_list[i]
            current_theta = self._get_theta_at_time(theta0, t)
            x_center, y_center = self._get_ball_position(current_theta)

            self.balls[i].center = (x_center, y_center)
            artists_to_update.append(self.balls[i])

        return artists_to_update

    def animate(self):
        """Creates and runs (or saves) the animation."""
        self._setup_plot()
        if self.fig is None:
            raise ValueError("Figure was not properly initialized")
        ani = FuncAnimation(
            self.fig,
            self._update,
            frames=self.num_frames,
            interval=int(1000 / self.fps),
            blit=False,
            repeat=True,
        )
        if self.save_anim:
            if not self.filename:
                angles_str = "_".join(map(str, self.initial_angles_deg))
                self.filename = f"tautochrone_{self.num_balls}balls_{angles_str}deg.gif"
            save_dir = "ANIMATIONS/TAUTOCHRONE"
            os.makedirs(save_dir, exist_ok=True)
            filepath = os.path.join(save_dir, self.filename)
            print(f"Saving animation to {os.path.abspath(filepath)}...")
            try:
                if self.filename.lower().endswith(".gif"):
                    writer = PillowWriter(fps=self.fps)
                else:
                    writer = FFMpegWriter(
                        fps=self.fps,
                        codec="libx264",
                        bitrate=8000,
                        extra_args=[
                            "-pix_fmt",
                            "yuv420p",
                            "-profile:v",
                            "high",
                            "-level",
                            "4.1",
                            "-movflags",
                            "+faststart",
                        ],
                    )
                ani.save(filepath, writer=writer)
                print("Animation saved successfully!")
            except Exception as e:
                print(f"Error saving animation: {e}")
                print("Make sure you have necessary writers installed (e.g., Pillow).")
                print("Attempting to show animation instead.")
                plt.show()
            finally:
                plt.close(self.fig)
        else:
            plt.show()
        return ani


if __name__ == "__main__":
    animator = TautochroneAnimator(
        initial_angles_deg=[0, 90, 135, 225, 270, 360],
        r=10,
        ball_radius=1,
        g=10,
        fps=50,
        # save_anim=True
    )
    anim = animator.animate()

### Animating the Isochronous motion in 3D

In [ ]:
class TautochroneAnimator3D:
    """
    Animates the tautochrone problem in 3D, showing particles sliding down
    cycloidal paths from different starting points with one ball per path.
    """

    def __init__(
        self,
        initial_angles_deg,
        r=1.0,
        ball_radius=0.1,
        g=9.81,
        fps=30,
        save_anim=False,
        filename=None,
        cuboid_padding=0.2,
    ):
        self.r = r
        self.g = g
        self.fps = fps
        self.save_anim = save_anim
        self.filename = filename
        self.ball_radius = ball_radius
        self.cuboid_padding = cuboid_padding

        if isinstance(initial_angles_deg, (int, float)):
            self.initial_angles_deg = [float(initial_angles_deg)]
        elif isinstance(initial_angles_deg, list):
            self.initial_angles_deg = [float(angle) for angle in initial_angles_deg]
        else:
            raise TypeError("initial_angles_deg must be a number or a list of numbers.")

        if 180.0 not in self.initial_angles_deg:
            self.initial_angles_deg.append(180.0)
            self.has_stationary_ball = True
        else:
            self.has_stationary_ball = True

        if not all(0 <= angle <= 360 for angle in self.initial_angles_deg):
            raise ValueError("Initial angles must be between 0 and 360 degrees.")

        self.theta0_list = [np.radians(angle) for angle in self.initial_angles_deg]
        self.num_paths = len(self.initial_angles_deg)
        self.num_balls = self.num_paths

        self.omega = np.sqrt(self.g / (4 * self.r))
        self.time_to_bottom = np.pi / (2 * self.omega)

        self.duration = self.time_to_bottom * 1.2
        self.num_frames = int(self.duration * self.fps)

        self.fig = None
        self.ax = None
        self.balls = []
        self.ball_colors = []
        self.paths = []

        self.stationary_idx = self.initial_angles_deg.index(180.0)

        # Pre-calculate a base 3D sphere to be translated later
        self.u = np.linspace(0, 2 * np.pi, 15)
        self.v = np.linspace(0, np.pi, 15)
        self.x_base = self.ball_radius * np.outer(np.cos(self.u), np.sin(self.v))
        self.y_base = self.ball_radius * np.outer(np.sin(self.u), np.sin(self.v))
        self.z_base = self.ball_radius * np.outer(
            np.ones(np.size(self.u)), np.cos(self.v)
        )

    def _cycloid_path(self, theta):
        """Parametric equation for the inverted cycloid path in x-z plane."""
        x = self.r * (theta - np.sin(theta))
        z = -self.r * (1 - np.cos(theta))
        return x, z

    def _get_theta_at_time(self, theta0, t, is_stationary=False):
        """Calculates the angular position theta(t) of a ball."""
        if is_stationary:
            return np.pi

        if t >= self.time_to_bottom:
            return np.pi
        elif t <= 0:
            return theta0
        else:
            omega = self.omega
            cos_val = np.cos(omega * t)
            if theta0 < np.pi:
                arg = np.cos(theta0 / 2.0) * cos_val
                arg = np.clip(arg, -1.0, 1.0)
                return 2.0 * np.arccos(arg)
            else:
                k = -np.cos(theta0 / 2.0)
                arg = k * cos_val
                arg = np.clip(arg, -1.0, 1.0)
                return 2.0 * np.pi - 2.0 * np.arccos(arg)

    def _get_ball_position(self, theta, y_offset):
        """
        Calculates the center position for the ball in 3D so it physically rests ON the cycloid curve.
        """
        x_curve, z_curve = self._cycloid_path(theta)

        if np.isclose(theta, 0) or np.isclose(theta, 2 * np.pi):
            nx = 1 if theta < np.pi else -1
            nz = 0
        elif np.isclose(theta, np.pi):
            nx = 0
            nz = 1
        else:
            nx = self.r * np.sin(theta)
            nz = self.r * (1 - np.cos(theta))
            norm = np.sqrt(nx**2 + nz**2)
            if norm > 1e-9:
                nx /= norm
                nz /= norm

        x_center = x_curve + self.ball_radius * nx
        z_center = z_curve + self.ball_radius * nz
        y_center = y_offset

        return x_center, y_center, z_center

    def _setup_plot(self):
        """Sets up the 3D matplotlib figure and axes."""
        plt.style.use("dark_background")
        self.fig = plt.figure(figsize=(10, 8))
        self.fig.tight_layout()

        self.ax = self.fig.add_subplot(111, projection="3d")
        self.fig.subplots_adjust(left=0.01, right=0.98, top=0.92, bottom=0.05)
        self.ax.set_title(
            f"3D Tautochrone Curves (Cycloids)\nTime to bottom: {self.time_to_bottom:.3f}s",
            fontsize=16,
        )

        max_x = self.r * (2 * np.pi)
        cuboid_length = max_x
        cuboid_width = self.r * self.num_paths * (1 + self.cuboid_padding)
        cuboid_height = 2 * self.r * (1 + self.cuboid_padding)

        self.ax.set_xlim(0, cuboid_length)
        self.ax.set_ylim(0, cuboid_width)
        self.ax.set_zlim(  # type: ignore
            -2 * self.r - self.r * self.cuboid_padding, self.r * self.cuboid_padding
        )

        self.ax.set_box_aspect(
            aspect=(cuboid_length, cuboid_width, cuboid_height),
            zoom=1.2,  # type: ignore
        )

        # Remove gridlines and ticks
        self.ax.axis("off")
        for axis in ["x", "y", "z"]:
            getattr(self.ax, f"{axis}axis").set_pane_color((0, 0, 0, 1))
        self.ax.grid(False)

        # Create cuboid
        vertices = np.array(
            [
                [0, 0, -2 * self.r],
                [cuboid_length, 0, -2 * self.r],
                [cuboid_length, cuboid_width, -2 * self.r],
                [0, cuboid_width, -2 * self.r],
                [0, 0, -2 * self.r + cuboid_height],
                [cuboid_length, 0, -2 * self.r + cuboid_height],
                [cuboid_length, cuboid_width, -2 * self.r + cuboid_height],
                [0, cuboid_width, -2 * self.r + cuboid_height],
            ]
        )

        faces = [
            [vertices[0], vertices[1], vertices[2], vertices[3]],
            [vertices[4], vertices[5], vertices[6], vertices[7]],
            [vertices[0], vertices[1], vertices[5], vertices[4]],
            [vertices[2], vertices[3], vertices[7], vertices[6]],
            [vertices[1], vertices[2], vertices[6], vertices[5]],
            [vertices[0], vertices[3], vertices[7], vertices[4]],
        ]

        cuboid = Poly3DCollection(faces, alpha=0.2, linewidth=1, edgecolor="white")
        cuboid.set_facecolor("gray")
        cuboid.set_zorder(1)
        self.ax.add_collection3d(cuboid)  # type: ignore

        # Plot cycloid paths and initialize balls
        theta_path = np.linspace(0, 2 * np.pi, 400)
        x_path, z_path = self._cycloid_path(theta_path)
        color_cycle = itertools.cycle(plt.cm.tab10.colors)  # type: ignore

        proxy_legend_artists = []

        for i in range(self.num_paths):
            y_offset = (i + 0.5) * (cuboid_width / self.num_paths)
            self.ax.plot(
                x_path, [y_offset] * len(x_path), z_path, color="gray", lw=2.5, zorder=2
            )

            theta0 = self.theta0_list[i]
            color = next(color_cycle)
            self.ball_colors.append(color)

            x0, y0, z0 = self._get_ball_position(theta0, y_offset)

            if i == self.stationary_idx:
                label = "Stationary Ball (180°)"
            else:
                label = f"Ball {i + 1}: {np.degrees(theta0):.0f}°"

            # Plot balls as 3D Spheres
            ball_surf = self.ax.plot_surface(  # type: ignore
                self.x_base + x0,
                self.y_base + y0,
                self.z_base + z0,
                color=color,
                zorder=5,
            )
            self.balls.append(ball_surf)
            proxy = Line2D(
                [0],
                [0],
                linestyle="none",
                marker="o",
                markersize=10,
                markerfacecolor=color,
                markeredgecolor=color,
                label=label,
            )
            proxy_legend_artists.append(proxy)

        # Add equation text
        cycloid_eq = "Inverted Cycloid: $x = r\\,(\\theta - \\sin\\theta)$, $z = -r\\,(1 - \\cos\\theta)$ ; $(0 \\leq \\theta \\leq 2\\pi)$"
        self.ax.text2D(  # type: ignore
            0.5,
            0.97,
            cycloid_eq,
            transform=self.ax.transAxes,
            ha="center",
            va="top",
            fontsize=10,
            color="white",
            bbox=dict(
                facecolor="black", alpha=0.8, edgecolor="gray", boxstyle="round,pad=0.3"
            ),
        )

        self.ax.legend(
            handles=proxy_legend_artists,
            bbox_to_anchor=(0.5, -0.02),
            loc="lower center",
            ncols=2,
            fontsize=8,
        )

    def _update(self, frame):
        """Updates the animation elements for each frame."""
        t = frame / self.fps
        artists_to_update = []
        for ball_surf in self.balls:
            ball_surf.remove()
        self.balls.clear()

        for i in range(self.num_paths):
            y_offset = (i + 0.5) * (
                self.r * self.num_paths * (1 + self.cuboid_padding) / self.num_paths
            )

            is_stationary = i == self.stationary_idx
            theta0 = self.theta0_list[i]

            current_theta = self._get_theta_at_time(theta0, t, is_stationary)
            x, y, z = self._get_ball_position(current_theta, y_offset)

            # Redraw sphere at new location
            surf = self.ax.plot_surface(  # type: ignore
                self.x_base + x,
                self.y_base + y,
                self.z_base + z,
                color=self.ball_colors[i],
                zorder=5,
            )

            self.balls.append(surf)
            artists_to_update.append(surf)

        elevs = np.linspace(5, 10, self.num_frames)
        azims = np.linspace(250, 280, self.num_frames)
        self.ax.view_init(elev=elevs[frame], azim=azims[frame])  # type: ignore

        return artists_to_update

    def animate(self):
        """Creates and runs (or saves) the animation."""
        self._setup_plot()
        if self.fig is None:
            raise ValueError("Figure was not properly initialized")
        ani = FuncAnimation(
            self.fig,
            self._update,
            frames=self.num_frames,
            interval=int(1000 / self.fps),
            blit=False,
            repeat=False,
        )
        if self.save_anim:
            if not self.filename:
                angles_str = "_".join(map(str, self.initial_angles_deg))
                self.filename = (
                    f"tautochrone_3d_{self.num_balls}balls_{angles_str}deg.gif"
                )
            save_dir = "ANIMATIONS/TAUTOCHRONE"
            os.makedirs(save_dir, exist_ok=True)
            filepath = os.path.join(save_dir, self.filename)
            print(f"Saving animation to {os.path.abspath(filepath)}...")
            try:
                if self.filename.lower().endswith(".gif"):
                    writer = PillowWriter(fps=self.fps)
                else:
                    writer = FFMpegWriter(
                        fps=self.fps,
                        codec="libx264",
                        bitrate=8000,
                        extra_args=[
                            "-pix_fmt",
                            "yuv420p",
                            "-profile:v",
                            "high",
                            "-level",
                            "4.1",
                            "-movflags",
                            "+faststart",
                        ],
                    )
                ani.save(filepath, writer=writer)
                print("Animation saved successfully!")
            except Exception as e:
                print(f"Error saving animation: {e}")
                print("Make sure you have necessary writers installed (e.g., Pillow).")
                print("Attempting to show animation instead.")
                plt.show()
            finally:
                plt.close(self.fig)
        else:
            plt.show()
        return ani


if __name__ == "__main__":
    print("\nRunning 3D animation with one ball per path...")
    animator = TautochroneAnimator3D(
        initial_angles_deg=[0, 90, 135, 180, 225, 270, 360],
        r=10,
        ball_radius=1.6,
        g=9.8,
        fps=60,
        save_anim=True,
        filename="test2.mp4",
    )
    anim = animator.animate()


Running 3D animation with one ball per path...
Saving animation to d:\VS CODE FILES\PhysicsXPython\Brachistochrone\ANIMATIONS\TAUTOCHRONE\test2.mp4...
Animation saved successfully!


## References
 
- https://www.internationaljournalssrg.org/IJAP/2015/Volume2-Issue3/IJAP-V2I5P101.pdf
- [The Brachistochrone: A Mathematical Journey Through Time and Space](https://medium.com/@priyanshu.pansari/the-brachistochrone-a-mathematical-journey-through-time-and-space-fd8c921e891e)
- [Scipython Blog: The Brachistochrone Problem](https://scipython.com/blog/the-brachistochrone-problem/)